# ==================================================
# SSN NEURALFORGE — FINAL REVIEW
# ==================================================
**IEEE BigData 2026 — Explainable Suicide Risk Detection Challenge**

This notebook is the FINAL REVIEW version. It trains pretrained Transformers on GPU, compares them with the
first-review TF-IDF models on the **same fixed leakage-safe validation split**, selects the best model by a
transparent policy, retrains on all labelled data, and writes the submission CSV.

**Integrity rules enforced in code**
* The first-review numbers (0.6468 / 0.5599 / 0.4419) are frozen constants and are never overwritten.
* Every "final" number is computed from validation predictions produced by a model that was executed in this run.
* `final_prediction_input.xlsx` is never used for fitting, tuning or model selection.
* If a model fails to run (OOM, download failure) it is logged as FAILED, never silently replaced by a made-up score.

**Sections:** 1 Overview · 2 Loading · 3 Statistics · 4 Quality · 5 Leakage · 6 First-review baseline · 7 Preprocessing ·
8 Split · 9 Baseline models · 10 Transformers · 11 Hyper-parameter search · 12 Risk results · 13 Risk ensemble ·
14 Evidence · 15 Factors · 16 Thresholds · 17 Factor ensemble · 18 Comparison · 19 First vs Final · 20 Selection ·
21 Retraining · 22 Prediction · 23 Validation · 24 Summary

## 1. Project Overview

| Task | First review (frozen) | Final-review approach |
|---|---|---|
| Risk (4-class) | TF-IDF word+char + Logistic Regression | DistilBERT / RoBERTa / DeBERTa-v3, class-weighted or focal loss, blend with TF-IDF LR |
| Evidence | TF-IDF LR candidate scoring (sentences + clauses) | Risk-aware gating + Transformer cross-encoder candidate scoring + hybrid with TF-IDF, exact-substring output |
| Factors (24, multi-label) | TF-IDF + OvR LR + per-factor thresholds | Multi-label Transformer (sigmoid, pos-weighted BCE / focal), per-factor thresholds 0.10–0.90, blend with TF-IDF LR |

### Algorithm 1 — Validated model selection (SSN Algorithm Specification)
```
Algorithm   : SelectBestModel
Input       : C = {c1..cn} candidate models, each with validation score s(c) from the fixed primary split,
              complexity rank r(c), tolerance tau = 0.01
Output      : c* the selected model
Variables   : s_max, E (eligible set)
Steps:
  1. s_max <- max over c in C of s(c)
  2. E <- { c in C : s(c) >= s_max - tau }
  3. c* <- element of E with the smallest r(c)   (ties: larger s(c))
  4. Print s(c) for all c, s_max, E and c*
  5. Return c*
```

In [ ]:
# ============ SETUP ============
import os, sys, re, ast, json, time, math, copy, random, gc, platform, warnings, pickle, subprocess
from collections import Counter
warnings.filterwarnings("ignore")

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import torch, transformers
except ImportError:
    _pip("torch", "transformers"); import torch, transformers
for _mod, _pkg in [("sentencepiece", "sentencepiece"), ("tiktoken", "tiktoken"), ("google.protobuf", "protobuf"),
                   ("openpyxl", "openpyxl"), ("reportlab", "reportlab")]:
    try: __import__(_mod)
    except ImportError: _pip(_pkg)

import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg") if os.environ.get("SSN_SMOKE") == "1" else None
import matplotlib.pyplot as plt
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import sklearn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             classification_report, hamming_loss, average_precision_score)
from scipy.sparse import hstack
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 200)

# ============ CONFIG ============
SMOKE = os.environ.get("SSN_SMOKE", "0") == "1"      # tiny random models for pipeline testing only; NEVER for results
SEED = 42
PRIMARY_SPLIT_SEED = 42                                # same split as the first review -> directly comparable
CLASSICAL_ROBUST_SEEDS = [42, 52, 62, 72, 82]
TRANSFORMER_ROBUST_SEEDS = [52, 62]                    # extra group splits for the selected Transformer configs
USE_CACHE = True                                       # reuse finished runs if the session restarts
SELECTION_TOLERANCE = float(os.environ.get("SSN_TOL", 0.01))                            # "very close -> prefer simpler model"
MAX_LEN = 256
AMP = True
RISK_GRID = [  # (model, [learning rates], micro_batch, grad_accum)  -> effective batch 16
    ("distilbert-base-uncased",   [3e-5, 5e-5], 16, 1),
    ("roberta-base",              [2e-5, 3e-5], 16, 1),
    ("microsoft/deberta-v3-base", [2e-5, 3e-5],  8, 2),
]
RISK_EPOCHS, RISK_PATIENCE = 8, 3
FACTOR_GRID = [
    ("distilbert-base-uncased",   [5e-5], 16, 1),
    ("roberta-base",              [3e-5], 16, 1),
    ("microsoft/deberta-v3-base", [3e-5],  8, 2),
]
FACTOR_EPOCHS, FACTOR_PATIENCE, FACTOR_HEAD_LR_MULT = 15, 4, 10.0
EVID_EPOCHS, EVID_PATIENCE, EVID_LR = 4, 2, 3e-5
if SMOKE:
    RISK_GRID = [("tiny-a", [1e-3], 16, 1), ("tiny-b", [2e-3], 16, 1)]
    FACTOR_GRID = [("tiny-a", [2e-3], 16, 1)]
    RISK_EPOCHS = FACTOR_EPOCHS = EVID_EPOCHS = 2; TRANSFORMER_ROBUST_SEEDS = [52]; MAX_LEN = 64
    CLASSICAL_ROBUST_SEEDS = [42, 52]

# ============ FIRST REVIEW BASELINE (FROZEN - DO NOT EDIT) ============
from types import MappingProxyType
BASELINE = MappingProxyType({"risk_weighted_f1": 0.6468, "evidence_phrase_f1": 0.5599, "factor_macro_f1": 0.4419})

# ============ DEVICE ============
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "none (CPU)"
GPU_MEM = f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB" if DEVICE.type == "cuda" else "n/a"

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ============ PATHS (local / Colab / Kaggle) ============
def find_file(name):
    cands = ["../data", "data", ".", "/content", "/content/data", "/content/drive/MyDrive"]
    if os.path.isdir("/kaggle/input"):
        for r, _, fs in os.walk("/kaggle/input"):
            if name in fs: return os.path.join(r, name)
    for d in cands:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(f"{name} not found. Place it in ../data/ (or upload to /content, or add as a Kaggle dataset).")

if os.environ.get("SSN_OUT"): OUT_DIR = os.environ["SSN_OUT"]
elif os.path.isdir("/kaggle/working"): OUT_DIR = "/kaggle/working/SSN_NeuralForge_FINAL/outputs"
elif os.path.isdir("../data"): OUT_DIR = "../outputs"
else: OUT_DIR = "outputs"
CODE_DIR = "." if not os.path.isdir("/kaggle/working") else "/kaggle/working/SSN_NeuralForge_FINAL/code"
REPORT_DIR = os.path.join(os.path.dirname(os.path.abspath(OUT_DIR)), "report")
CACHE_DIR = os.path.join(OUT_DIR, "cache" + ("_smoke" if SMOKE else ""))
for d in (OUT_DIR, CACHE_DIR, REPORT_DIR, CODE_DIR): os.makedirs(d, exist_ok=True)

def cached(key, fn):
    path = os.path.join(CACHE_DIR, re.sub(r"[^A-Za-z0-9_.-]", "_", key) + ".pkl")
    if USE_CACHE and os.path.exists(path):
        print(f"   [cache hit] {key}"); return pickle.load(open(path, "rb"))
    out = fn()
    if not (isinstance(out, dict) and "failed" in out): pickle.dump(out, open(path, "wb"))     # never cache failures
    return out

ENV = {"python": sys.version.split()[0], "torch": torch.__version__, "transformers": transformers.__version__,
       "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__, "gpu": GPU_NAME,
       "gpu_memory": GPU_MEM, "device": str(DEVICE), "seed": SEED, "max_len": MAX_LEN, "amp": bool(AMP and DEVICE.type == "cuda")}
print("=" * 60); print("ENVIRONMENT"); print("=" * 60)
for k, v in ENV.items(): print(f"{k:14s}: {v}")
if SMOKE: print("\n!!! SMOKE MODE: tiny random models - numbers are meaningless, pipeline test only !!!")
elif DEVICE.type != "cuda":
    print("\nWARNING: no GPU. Runtime -> Change runtime type -> GPU (T4). Transformer training on CPU is impractical.")

## 2. Dataset Loading

In [ ]:
train_df = pd.read_excel(find_file("train.xlsx"))
test_df = pd.read_excel(find_file("final_prediction_input.xlsx"))   # UNLABELLED. Not used for anything until Section 22.
print("train_df:", train_df.shape, "| prediction input:", test_df.shape)
print("train columns:", list(train_df.columns)); print("test columns :", list(test_df.columns))

## 3. Dataset Statistics

In [ ]:
RISK_ORDER = ["Indicator", "Ideation", "Behavior", "Attempt"]
train_df["risk_level"] = train_df["suicide risk"].astype(str).str.strip().str.capitalize()
assert set(train_df["risk_level"]) == set(RISK_ORDER), set(train_df["risk_level"])

def parse_factors(v):
    try:
        p = ast.literal_eval(v)
        if isinstance(p, list): return p
    except (ValueError, SyntaxError): pass
    return []
train_df["factors_unique"] = train_df["factors"].apply(lambda v: sorted(set(parse_factors(v))))
OFFICIAL_FACTORS = sorted({
    "mental health issues", "physical health/characteristic", "substance use", "hopelessness", "emotion dysregulation",
    "low self-esteem", "poor school performance", "low socio-economic status", "interpersonal violence",
    "prior self-harm or suicidal thought/attempt", "poor social support", "interpersonal difficulty", "dysfunctional family",
    "exposure to others' suicide", "stressful life event", "traumatic experience", "cognitive deficits",
    "suicide means (with access)", "sexual orientation related issues", "social support", "coping strategy",
    "psychological capital", "sense of responsibility", "meaning in life"})
assert set(f for l in train_df["factors_unique"] for f in l) <= set(OFFICIAL_FACTORS)
N_FACTORS = len(OFFICIAL_FACTORS)

print("Posts:", len(train_df), "| users:", train_df.anon_user_id.nunique(), "| prediction posts:", len(test_df), "| prediction users:", test_df.anon_user_id.nunique())
print("\nRisk distribution:\n", train_df.risk_level.value_counts().reindex(RISK_ORDER).to_string())
fc = Counter(f for l in train_df.factors_unique for f in l)
print("\nFactor frequency (posts):"); print(pd.Series(fc).sort_values(ascending=False).to_string())
wc = train_df.post.astype(str).str.split().str.len()
print("\nWords per post:", wc.describe().round(1).to_dict())
ev_gold_none = train_df["evidence for suicide risk level"].astype(str).str.strip().str.lower().eq("none")
print("\nEvidence == 'none' per risk level:\n", train_df.assign(n=ev_gold_none).groupby("risk_level")["n"].mean().round(3).reindex(RISK_ORDER).to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
train_df.risk_level.value_counts().reindex(RISK_ORDER).plot.bar(ax=ax[0], color="#4C72B0"); ax[0].set_title("Risk distribution"); ax[0].set_ylabel("posts")
pd.Series(fc).sort_values().plot.barh(ax=ax[1], color="#55A868"); ax[1].set_title("Factor frequency"); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "dataset_statistics.png"), dpi=120); plt.show()

## 4. Data Quality Checks

In [ ]:
qc = {"missing post": int(train_df.post.isna().sum()), "empty post": int((train_df.post.astype(str).str.strip() == "").sum()),
      "duplicate row_id": int(train_df.row_id.duplicated().sum()), "duplicate post text": int(train_df.post.duplicated().sum()),
      "raw risk label spellings": int(train_df["suicide risk"].nunique()), "normalised risk labels": int(train_df.risk_level.nunique()),
      "posts without factors": int((train_df.factors_unique.str.len() == 0).sum()),
      "missing evidence": int(train_df["evidence for suicide risk level"].isna().sum()),
      "prediction: missing post": int(test_df.post.isna().sum()), "prediction: duplicate row_id": int(test_df.row_id.duplicated().sum())}
print(pd.Series(qc).to_string())

## 5. DATA LEAKAGE CHECK

In [ ]:
_n = lambda t: re.sub(r"\s+", " ", str(t)).strip().lower()
ds_rows = [("user overlap  train.xlsx ∩ prediction file", len(set(train_df.anon_user_id) & set(test_df.anon_user_id))),
           ("row_id overlap train.xlsx ∩ prediction file", len(set(train_df.row_id) & set(test_df.row_id))),
           ("identical post text train.xlsx ∩ prediction file", len(set(train_df.post.map(_n)) & set(test_df.post.map(_n))))]
print(pd.DataFrame(ds_rows, columns=["check", "count (must be 0 for users / row_ids)"]).to_string(index=False))
assert ds_rows[0][1] == 0 and ds_rows[1][1] == 0, "LEAKAGE DETECTED between train and prediction file"
print("\nPolicy (enforced by construction, verified by this notebook):")
for s_ in ["Validation labels are used only for evaluation, early stopping, and threshold/blend/decoder tuning of the model being validated.",
           "The prediction file is read into test_df but is not touched by any fit/tune/select call before Section 21.",
           "All splits are grouped by anon_user_id; no user appears on both sides (verified again after the split in Section 8).",
           "Final retraining on all labelled data happens only AFTER every validation number is frozen in RESULTS (Section 20).",
           "No test/leaderboard feedback is used anywhere."]: print("  *", s_)
if ds_rows[2][1]: print(f"\nNOTE: {ds_rows[2][1]} identical post text(s) appear in both train.xlsx and the prediction file (different users). Reported, not removed.")

## 6. FIRST REVIEW BASELINE  (frozen constants + reproduction check)

In [ ]:
print("=" * 50); print("FIRST REVIEW BASELINE"); print("=" * 50)
print(f"Risk Classification       : {BASELINE['risk_weighted_f1']:.4f} Weighted F1")
print(f"Evidence Extraction       : {BASELINE['evidence_phrase_f1']:.4f} Phrase F1")
print(f"Factor Identification     : {BASELINE['factor_macro_f1']:.4f} Macro F1")
print("=" * 50)
RESULTS = {"baseline": dict(BASELINE), "env": ENV, "candidates": {}, "robustness": {}, "bootstrap": {}}

## 7. Preprocessing  (executed before the split so every split uses the same cleaned text)

In [ ]:
def clean_text(t):
    if pd.isna(t): return ""
    return re.sub(r"\s+", " ", str(t)).strip()           # minimal: keep punctuation + negations
for d in (train_df, test_df):
    d["post_clean"] = d["post"].apply(clean_text); d["post_clean_lower"] = d["post_clean"].str.lower()
print("Empty cleaned posts (train / prediction):", (train_df.post_clean == "").sum(), "/", (test_df.post_clean == "").sum())

## 8. Train / Validation Split  (fixed primary split = first review split)

In [ ]:
def get_group_split(seed):
    sp = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr, va = next(sp.split(train_df, groups=train_df["anon_user_id"]))
    return train_df.iloc[tr].reset_index(drop=True), train_df.iloc[va].reset_index(drop=True)

train_split, val_split = get_group_split(PRIMARY_SPLIT_SEED)
print("Train rows/users:", len(train_split), train_split.anon_user_id.nunique(), "| Val rows/users:", len(val_split), val_split.anon_user_id.nunique())
mlb = MultiLabelBinarizer(classes=OFFICIAL_FACTORS)
Y_TR = mlb.fit_transform(train_split.factors_unique); Y_VA = mlb.transform(val_split.factors_unique)
y_tr_risk = train_split.risk_level.map({r: i for i, r in enumerate(RISK_ORDER)}).values
y_va_risk = val_split.risk_level.map({r: i for i, r in enumerate(RISK_ORDER)}).values
print("Factors with zero positives in primary train / val split:", int((Y_TR.sum(0) == 0).sum()), "/", int((Y_VA.sum(0) == 0).sum()))

In [ ]:
def leakage_report():
    rows = []
    tr_u, va_u = set(train_split.anon_user_id), set(val_split.anon_user_id)
    rows.append(("user overlap  train_split ∩ val_split (primary)", len(tr_u & va_u)))
    rows.append(("row_id overlap train_split ∩ val_split", len(set(train_split.row_id) & set(val_split.row_id))))
    rows.append(("identical post text train_split ∩ val_split (informational)", len(set(train_split.post_clean) & set(val_split.post_clean))))
    for sd in CLASSICAL_ROBUST_SEEDS + TRANSFORMER_ROBUST_SEEDS:
        a, b = get_group_split(sd); rows.append((f"user overlap robustness split seed={sd}", len(set(a.anon_user_id) & set(b.anon_user_id))))
    return pd.DataFrame(rows, columns=["check", "count"])
leak = leakage_report(); print("DATA LEAKAGE CHECK (split level)"); print(leak.to_string(index=False))
assert leak[~leak.check.str.contains("informational")]["count"].sum() == 0, "LEAKAGE DETECTED"
print("-> no user or row shared between any training and validation split.")

## 9. Baseline Models  (first-review classical pipelines, reproduced on the primary split)

In [ ]:
def risk_features(tr_text, others):
    wv = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=10000)
    cv = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2, max_features=10000)
    Xt = hstack([wv.fit_transform(tr_text), cv.fit_transform(tr_text)]).tocsr()
    return Xt, [hstack([wv.transform(t), cv.transform(t)]).tocsr() for t in others], (wv, cv)

def risk_metrics(y_true_ids, pred_ids):
    p, r, f, s = precision_recall_fscore_support(y_true_ids, pred_ids, labels=range(4), zero_division=0)
    return {"weighted_f1": f1_score(y_true_ids, pred_ids, average="weighted"), "macro_f1": f1_score(y_true_ids, pred_ids, average="macro"),
            "accuracy": accuracy_score(y_true_ids, pred_ids), "per_class": pd.DataFrame({"precision": p, "recall": r, "f1": f, "support": s}, index=RISK_ORDER)}

def reorder_proba(model, X):
    pr = model.predict_proba(X); return pr[:, [list(model.classes_).index(i) for i in range(4)]]

# ---- classical risk (first-review recipe) ----
t0 = time.time()
Xtr_r, (Xva_r,), _ = risk_features(train_split.post_clean_lower, [val_split.post_clean_lower])
lr_risk = LogisticRegression(C=1, class_weight="balanced", max_iter=1000, random_state=SEED).fit(Xtr_r, y_tr_risk)
LR_VAL_PROBA = reorder_proba(lr_risk, Xva_r)
m_lr = risk_metrics(y_va_risk, LR_VAL_PROBA.argmax(1))
svm_risk = None; best_svm = (-1, None)
for C in [0.25, 0.5, 1, 2]:
    sv = LinearSVC(C=C, class_weight="balanced", random_state=SEED, max_iter=5000).fit(Xtr_r, y_tr_risk)
    f = f1_score(y_va_risk, sv.predict(Xva_r), average="weighted")
    if f > best_svm[0]: best_svm = (f, C, sv)
SVM_C = best_svm[1]; m_svm = risk_metrics(y_va_risk, best_svm[2].predict(Xva_r))
print(f"TF-IDF(word+char)+LogReg : Weighted F1 = {m_lr['weighted_f1']:.4f}   (first review: {BASELINE['risk_weighted_f1']:.4f})")
print(f"TF-IDF(word+char)+LinearSVM(C={SVM_C}) : Weighted F1 = {m_svm['weighted_f1']:.4f}")
RESULTS["repro_risk"] = m_lr["weighted_f1"]
if abs(m_lr["weighted_f1"] - BASELINE["risk_weighted_f1"]) > 0.003:
    print("NOTE: reproduced value differs from the frozen first-review number (library versions?). The FROZEN number remains the reference.")

# ---- classical risk robustness over 5 group splits ----
rob = {"LR": [], "SVM": []}
for s in CLASSICAL_ROBUST_SEEDS:
    a, b = get_group_split(s); ya = a.risk_level.map({r: i for i, r in enumerate(RISK_ORDER)}).values; yb = b.risk_level.map({r: i for i, r in enumerate(RISK_ORDER)}).values
    Xa, (Xb,), _ = risk_features(a.post_clean_lower, [b.post_clean_lower])
    rob["LR"].append(f1_score(yb, LogisticRegression(C=1, class_weight="balanced", max_iter=1000, random_state=SEED).fit(Xa, ya).predict(Xb), average="weighted"))
    rob["SVM"].append(f1_score(yb, LinearSVC(C=SVM_C, class_weight="balanced", random_state=SEED, max_iter=5000).fit(Xa, ya).predict(Xb), average="weighted"))
for k, v in rob.items(): print(f"{k}: per-split {np.round(v, 4)}  mean={np.mean(v):.4f} std={np.std(v):.4f}")
RESULTS["robustness"]["risk_LR"] = rob["LR"]; RESULTS["robustness"]["risk_SVM"] = rob["SVM"]

# ---- classical factors (first-review recipe) ----
vec_f = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=10000)
Xtr_f = vec_f.fit_transform(train_split.post_clean_lower); Xva_f = vec_f.transform(val_split.post_clean_lower)
def fit_ovr_lr(X, Y):
    # labels with no positive example in the training split cannot be fitted -> constant 0 probability
    cols = [j for j in range(Y.shape[1]) if Y[:, j].sum() > 0]
    clf = OneVsRestClassifier(LogisticRegression(C=1, class_weight="balanced", max_iter=1000, random_state=SEED)).fit(X, Y[:, cols])
    return clf, cols
def ovr_proba(clf_cols, X, n=N_FACTORS):
    clf, cols = clf_cols; out = np.zeros((X.shape[0], n)); out[:, cols] = clf.predict_proba(X); return out
lr_fac = fit_ovr_lr(Xtr_f, Y_TR); LR_FAC_VAL_PROBA = ovr_proba(lr_fac, Xva_f)

def tune_thr_old(y, p, grid=(0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6)):       # first-review threshold search, verbatim logic
    th = []
    for j in range(y.shape[1]):
        bt, bf = 0.5, -1
        for t in grid:
            f = f1_score(y[:, j], (p[:, j] >= t).astype(int), zero_division=0)
            if f > bf: bf, bt = f, t
        th.append(bt)
    return np.array(th)
th_old = tune_thr_old(Y_VA, LR_FAC_VAL_PROBA)
repro_fac = f1_score(Y_VA, (LR_FAC_VAL_PROBA >= th_old).astype(int), average="macro", zero_division=0)
print(f"\nTF-IDF + OvR LogReg + per-factor thresholds (first-review grid): Macro F1 = {repro_fac:.4f}   (first review: {BASELINE['factor_macro_f1']:.4f})")
RESULTS["repro_factor"] = repro_fac
print(f"classical time: {time.time()-t0:.1f}s")

## 10. Improved Transformer Models — training engine

### Algorithm 2 — Fine-tuning with early stopping (SSN Algorithm Specification)
```
Algorithm   : FineTuneTransformer
Input       : backbone B, training set D_tr, validation set D_va, learning rate eta, max epochs E, patience P,
              loss type L in {CE, weighted-CE, focal | BCE, pos-weighted BCE, focal}, seed s
Output      : best checkpoint theta*, best epoch e*, validation probabilities P_va, best validation score S*
Variables   : theta (weights), S (score), bad (epochs without improvement), sched (linear warm-up/decay schedule)
Steps:
  1. set all random seeds to s; load B with a fresh classification head (sigmoid head if multi-label)
  2. build AdamW with weight-decay 0.01 (no decay on bias/LayerNorm) and a linear schedule with 10 % warm-up
  3. S* <- -1 ; bad <- 0
  4. for e = 1 to E do
  5.     for each mini-batch (x, y) in D_tr do
  6.         compute loss under L (fp16 autocast); back-propagate; clip gradient norm to 1.0; optimiser + sched step
  7.     P <- predict(D_va) ; S <- metric(y_va, P)
  8.     if S > S* then S* <- S ; e* <- e ; theta* <- copy(theta) ; P_va <- P ; bad <- 0
  9.     else bad <- bad + 1 ; if bad >= P then break
 10. restore theta* ; return theta*, e*, P_va, S*
```

In [ ]:
# ---------- tokenizer / backbone loading (SMOKE mode swaps in tiny random models, for pipeline tests only) ----------
_TOK = {}
def load_tokenizer(name):
    if name in _TOK: return _TOK[name]
    if SMOKE:
        from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers, processors
        from transformers import PreTrainedTokenizerFast
        tk = Tokenizer(models.WordPiece(unk_token="[UNK]")); tk.normalizer = normalizers.Lowercase(); tk.pre_tokenizer = pre_tokenizers.Whitespace()
        tk.train_from_iterator(train_df.post_clean.tolist(), trainers.WordPieceTrainer(vocab_size=1500, special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]))
        tk.post_processor = processors.TemplateProcessing(single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1", special_tokens=[("[CLS]", tk.token_to_id("[CLS]")), ("[SEP]", tk.token_to_id("[SEP]"))])
        tok = PreTrainedTokenizerFast(tokenizer_object=tk, pad_token="[PAD]", unk_token="[UNK]", cls_token="[CLS]", sep_token="[SEP]", mask_token="[MASK]")
    else:
        tok = AutoTokenizer.from_pretrained(name)
    _TOK[name] = tok; return tok

def load_backbone(name, num_labels, problem_type):
    if SMOKE:
        from transformers import BertConfig, BertForSequenceClassification
        tok = load_tokenizer(name)
        cfg = BertConfig(vocab_size=len(tok), hidden_size=32, num_hidden_layers=2, num_attention_heads=2, intermediate_size=64,
                         max_position_embeddings=512, num_labels=num_labels, problem_type=problem_type)
        return BertForSequenceClassification(cfg)
    return AutoModelForSequenceClassification.from_pretrained(name, num_labels=num_labels, problem_type=problem_type)

def encode(tok, texts, max_len, texts_b=None):
    if texts_b is None: enc = tok(list(texts), truncation=True, max_length=max_len)
    else: enc = tok(list(texts), text_pair=list(texts_b), truncation="longest_first", max_length=max_len)
    return [{k: enc[k][i] for k in enc.keys()} for i in range(len(texts))]

class EncDS(Dataset):
    def __init__(self, feats, labels=None): self.f, self.y = feats, labels
    def __len__(self): return len(self.f)
    def __getitem__(self, i): return self.f[i], (self.y[i] if self.y is not None else 0)

def make_collate(tok, multilabel):
    def collate(items):
        b = tok.pad([f for f, _ in items], return_tensors="pt")
        y = torch.tensor(np.array([y for _, y in items]), dtype=torch.float32 if multilabel else torch.long)
        return b, y
    return collate

HEAD_KEYS = ("classifier", "pre_classifier", "pooler", "score")
def make_loss(task, loss_type, y_train):
    if task == "multiclass":
        k = int(y_train.max()) + 1; cnt = np.bincount(y_train, minlength=k).astype(float)
        w = torch.tensor(len(y_train) / (k * np.maximum(cnt, 1)), dtype=torch.float32, device=DEVICE)
        if loss_type == "ce": return lambda lg, y: F.cross_entropy(lg.float(), y)
        if loss_type == "wce": return lambda lg, y: F.cross_entropy(lg.float(), y, weight=w)
        if loss_type == "focal":
            def focal(lg, y):
                lp = F.log_softmax(lg.float(), -1); pt = lp.exp().gather(1, y[:, None]).squeeze(1)
                return (((1 - pt) ** 2) * F.nll_loss(lp, y, weight=w, reduction="none")).mean()
            return focal
    else:
        pos = y_train.sum(0); neg = len(y_train) - pos
        pw = torch.tensor(np.clip(np.sqrt(neg / np.maximum(pos, 1)), 1, 10), dtype=torch.float32, device=DEVICE)
        if loss_type == "bce": return lambda lg, y: F.binary_cross_entropy_with_logits(lg.float(), y)
        if loss_type == "wbce": return lambda lg, y: F.binary_cross_entropy_with_logits(lg.float(), y, pos_weight=pw)
        if loss_type == "focal":
            def focal(lg, y):
                lg = lg.float(); b = F.binary_cross_entropy_with_logits(lg, y, pos_weight=pw, reduction="none")
                p = torch.sigmoid(lg); pt = p * y + (1 - p) * (1 - y); return (((1 - pt) ** 2) * b).mean()
            return focal
    raise ValueError(loss_type)

@torch.no_grad()
def predict_proba(model, loader, multilabel):
    model.eval(); out = []
    for b, _ in loader:
        b = {k: v.to(DEVICE) for k, v in b.items()}
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(AMP and DEVICE.type == "cuda")):
            lg = model(**b).logits
        out.append((torch.sigmoid(lg.float()) if multilabel else torch.softmax(lg.float(), -1)).cpu().numpy())
    return np.concatenate(out)

def train_transformer(model_name, task, tr_texts, y_tr, *, va_texts=None, y_va=None, tr_b=None, va_b=None, test_texts=None, test_b=None,
                      lr, epochs, micro_bs, accum, loss_type, max_len, seed, patience, metric_fn, num_labels,
                      head_lr_mult=1.0, fixed_epochs=None, warmup=0.1, wd=0.01, clip=1.0):
    """Returns dict(best_epoch, best_score, val_proba, test_proba, history, seconds)."""
    t0 = time.time(); multilabel = task == "multilabel"; set_seed(seed)
    tok = load_tokenizer(model_name)
    model = load_backbone(model_name, num_labels, "multi_label_classification" if multilabel else "single_label_classification").to(DEVICE)
    col = make_collate(tok, multilabel)
    g = torch.Generator(); g.manual_seed(seed)
    tr_dl = DataLoader(EncDS(encode(tok, tr_texts, max_len, tr_b), y_tr), batch_size=micro_bs, shuffle=True, collate_fn=col, generator=g)
    va_dl = DataLoader(EncDS(encode(tok, va_texts, max_len, va_b), y_va), batch_size=micro_bs * 2, collate_fn=col) if va_texts is not None else None
    te_dl = DataLoader(EncDS(encode(tok, test_texts, max_len, test_b)), batch_size=micro_bs * 2, collate_fn=col) if test_texts is not None else None
    nd = ("bias", "LayerNorm.weight", "layer_norm.weight")
    groups = []
    for is_head in (False, True):
        for decay in (True, False):
            ps = [p for n, p in model.named_parameters() if (any(k in n for k in HEAD_KEYS) == is_head) and ((not any(x in n for x in nd)) == decay)]
            if ps: groups.append({"params": ps, "weight_decay": wd if decay else 0.0, "lr": lr * (head_lr_mult if is_head else 1.0)})
    opt = torch.optim.AdamW(groups, lr=lr)
    n_ep = fixed_epochs or epochs; steps = max(1, math.ceil(len(tr_dl) / accum) * n_ep)
    sched = get_linear_schedule_with_warmup(opt, int(warmup * steps), steps)
    use_amp = AMP and DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    loss_fn = make_loss(task, loss_type, np.asarray(y_tr))
    best = {"score": -1.0, "epoch": 0, "proba": None, "state": None}; bad = 0; hist = []
    for ep in range(1, n_ep + 1):
        model.train(); tot = 0.0; nb = 0; opt.zero_grad()
        for i, (b, y) in enumerate(tr_dl):
            b = {k: v.to(DEVICE) for k, v in b.items()}; y = y.to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                lg = model(**b).logits
            loss = loss_fn(lg, y) / accum
            scaler.scale(loss).backward(); tot += loss.item() * accum; nb += 1
            if (i + 1) % accum == 0 or (i + 1) == len(tr_dl):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        rec = {"epoch": ep, "train_loss": tot / max(nb, 1)}
        if va_dl is not None:
            pr = predict_proba(model, va_dl, multilabel); sc = float(metric_fn(y_va, pr)); rec["val_score"] = sc
            if sc > best["score"] + 1e-6:
                best.update(score=sc, epoch=ep, proba=pr, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()}); bad = 0
            else: bad += 1
            hist.append(rec); print(f"      ep{ep:2d} loss={rec['train_loss']:.4f} val={sc:.4f}  best={best['score']:.4f}@{best['epoch']}")
            if bad >= patience: break
        else: hist.append(rec)
    if va_dl is not None and best["state"] is not None: model.load_state_dict(best["state"])
    test_proba = predict_proba(model, te_dl, multilabel) if te_dl is not None else None
    out = {"model": model_name, "lr": lr, "epochs_max": n_ep, "batch": micro_bs * accum, "loss": loss_type, "max_len": max_len, "seed": seed,
           "best_epoch": best["epoch"] if va_dl is not None else n_ep, "best_score": best["score"] if va_dl is not None else None,
           "val_proba": best["proba"], "test_proba": test_proba, "history": hist, "seconds": time.time() - t0}
    del model, opt, best; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return out

def safe_run(label, fn):
    """Run a training job; on failure record FAILED (never invent a score)."""
    try: return fn()
    except Exception as e:
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
        print(f"   !! {label} FAILED: {type(e).__name__}: {str(e)[:200]}"); return {"failed": f"{type(e).__name__}: {str(e)[:160]}", "model": label}

def risk_metric_fn(y, pr): return f1_score(y, pr.argmax(1), average="weighted")
print("Engine ready on", DEVICE, "|", GPU_NAME, GPU_MEM)

## 11. Hyper-parameter Search — risk classification
Controlled comparison on the **fixed primary split**. For each (model, learning rate): weighted-CE loss, linear warm-up/decay,
weight-decay 0.01, grad-clip 1.0, fp16, early stopping (patience 3) with best-checkpoint restore. Afterwards a loss ablation
(CE / weighted-CE / focal) is run only on the best architecture.

In [ ]:
RISK_RUNS = []
print(f"GPU: {GPU_NAME} ({GPU_MEM})")
for name, lrs, mb, acc in RISK_GRID:
    for lr in lrs:
        key = f"risk_{name}_lr{lr}_wce_s{SEED}_sp{PRIMARY_SPLIT_SEED}"; print(f"\n>> RISK  {name}  lr={lr}  loss=wce")
        r = cached(key, lambda: safe_run(key, lambda: train_transformer(
            name, "multiclass", train_split.post_clean, y_tr_risk, va_texts=val_split.post_clean, y_va=y_va_risk, lr=lr, epochs=RISK_EPOCHS,
            micro_bs=mb, accum=acc, loss_type="wce", max_len=MAX_LEN, seed=SEED, patience=RISK_PATIENCE, metric_fn=risk_metric_fn, num_labels=4)))
        r.setdefault("model", name); r["mb"], r["acc"] = mb, acc
        if "failed" not in r: print(f"   done: best weighted-F1={r['best_score']:.4f} @epoch {r['best_epoch']}  time={r['seconds']:.0f}s")
        RISK_RUNS.append(r)
ok = [r for r in RISK_RUNS if "failed" not in r]
risk_grid_df = pd.DataFrame([{"Model": r["model"], "LR": r.get("lr"), "Epochs(max)": r.get("epochs_max"), "Batch": r.get("batch"), "Loss": r.get("loss"),
        "Val Weighted F1": r.get("best_score"), "Val Macro F1": f1_score(y_va_risk, r["val_proba"].argmax(1), average="macro") if "failed" not in r else None,
        "Accuracy": accuracy_score(y_va_risk, r["val_proba"].argmax(1)) if "failed" not in r else None, "Best epoch": r.get("best_epoch"),
        "Train time (s)": r.get("seconds"), "Status": r.get("failed", "ok")} for r in RISK_RUNS])
print("\n"); print(risk_grid_df.round(4).to_string(index=False))
assert ok or SMOKE, "No Transformer risk run succeeded - fix the environment (GPU / internet) and re-run; the final notebook must not report Transformers as run if they did not."

In [ ]:
# ---- loss ablation on the best architecture/LR (class-imbalance handling: only keep what validation supports) ----
best_cfg = max(ok, key=lambda r: r["best_score"]) if ok else None
if best_cfg:
    print("Best grid run:", best_cfg["model"], best_cfg["lr"], f"{best_cfg['best_score']:.4f}")
    for lt in ["ce", "focal"]:
        key = f"risk_{best_cfg['model']}_lr{best_cfg['lr']}_{lt}_s{SEED}_sp{PRIMARY_SPLIT_SEED}"; print(f"\n>> RISK loss ablation: {lt}")
        r = cached(key, lambda: safe_run(key, lambda: train_transformer(
            best_cfg["model"], "multiclass", train_split.post_clean, y_tr_risk, va_texts=val_split.post_clean, y_va=y_va_risk, lr=best_cfg["lr"], epochs=RISK_EPOCHS,
            micro_bs=best_cfg["mb"], accum=best_cfg["acc"], loss_type=lt, max_len=MAX_LEN, seed=SEED, patience=RISK_PATIENCE, metric_fn=risk_metric_fn, num_labels=4)))
        r.setdefault("model", best_cfg["model"]); r["mb"], r["acc"] = best_cfg["mb"], best_cfg["acc"]; RISK_RUNS.append(r)
    ab = pd.DataFrame([{"Loss": r.get("loss"), "Val Weighted F1": r.get("best_score"), "Status": r.get("failed", "ok")} for r in RISK_RUNS
                       if r.get("model") == best_cfg["model"] and r.get("lr") == best_cfg["lr"] or "failed" in r])
    print("\nLoss ablation (same model, same LR):"); print(ab.round(4).to_string(index=False))
ok = [r for r in RISK_RUNS if "failed" not in r]
# best run per architecture = highest validation weighted F1 (ties -> first run)
RISK_BEST_BY_ARCH = {}
for r in ok:
    if r["model"] not in RISK_BEST_BY_ARCH or r["best_score"] > RISK_BEST_BY_ARCH[r["model"]]["best_score"]: RISK_BEST_BY_ARCH[r["model"]] = r
print("\nBest configuration per architecture:")
for k, r in RISK_BEST_BY_ARCH.items(): print(f"  {k:28s} lr={r['lr']}  loss={r['loss']}  weighted-F1={r['best_score']:.4f}  epoch={r['best_epoch']}  {r['seconds']:.0f}s")

## 12–13. Risk Classification Results and Risk Ensemble  (blend weight chosen on validation only)

In [ ]:
def select_best(cands, tol=SELECTION_TOLERANCE, label="model"):
    """Algorithm 1. cands: dict name -> dict(score, rank). Returns (name, decision text)."""
    smax = max(c["score"] for c in cands.values())
    elig = {n: c for n, c in cands.items() if c["score"] >= smax - tol}
    pick = sorted(elig, key=lambda n: (elig[n]["rank"], -elig[n]["score"]))[0]
    best_name = max(cands, key=lambda n: cands[n]["score"])
    txt = [f"SELECTION POLICY ({label}): highest validated score; if within {tol} of the best, prefer the simpler model.",
           f"  best raw score : {best_name} = {smax:.4f}", f"  eligible (within tolerance): {', '.join(sorted(elig, key=lambda n: elig[n]['rank']))}",
           f"  >>> SELECTED   : {pick}  (score {cands[pick]['score']:.4f}, complexity rank {cands[pick]['rank']})"]
    if pick != best_name: txt.append(f"  note: {pick} chosen over the numerically best {best_name} because the gap ({smax - cands[pick]['score']:.4f}) is within tolerance and it is simpler.")
    return pick, "\n".join(txt)

def blend_grid(pa, pb, score_fn):
    """pa weight w, pb weight 1-w, w in 0.0..1.0 step 0.1 ; returns (best_w, best_score, table)."""
    tab = []
    for w in np.round(np.linspace(0, 1, 11), 2): tab.append((w, score_fn(w * pa + (1 - w) * pb)))
    bw, bs = max(tab, key=lambda t: t[1]); return bw, bs, tab

ARCH_RANK = {"distilbert-base-uncased": 2, "roberta-base": 3, "microsoft/deberta-v3-base": 4}
RISK_CANDS = {}      # name -> {score, rank, proba, kind, components}
RISK_CANDS["TF-IDF + Logistic Regression"] = dict(score=m_lr["weighted_f1"], rank=0, proba=LR_VAL_PROBA, pred=LR_VAL_PROBA.argmax(1), kind="classical", comp={"LR": 1.0})
RISK_CANDS["TF-IDF + Linear SVM"] = dict(score=m_svm["weighted_f1"], rank=1, proba=None, pred=best_svm[2].predict(Xva_r), kind="classical", comp={"SVM": 1.0})
for arch, r in RISK_BEST_BY_ARCH.items():
    RISK_CANDS[arch] = dict(score=r["best_score"], rank=ARCH_RANK.get(arch, 2), proba=r["val_proba"], pred=r["val_proba"].argmax(1), kind="transformer", comp={arch: 1.0}, run=r)
failed_archs = [r["model"] for r in RISK_RUNS if "failed" in r and r["model"] not in RISK_BEST_BY_ARCH]

sf = lambda p: f1_score(y_va_risk, p.argmax(1), average="weighted")
tr_archs = list(RISK_BEST_BY_ARCH)
if tr_archs:
    top_arch = max(tr_archs, key=lambda a: RISK_BEST_BY_ARCH[a]["best_score"])
    w, s, tab = blend_grid(LR_VAL_PROBA, RISK_BEST_BY_ARCH[top_arch]["val_proba"], sf)    # w = weight of the CLASSICAL model
    print(f"Blend  LR(w) + {top_arch}(1-w):  " + "  ".join(f"{a:.1f}:{b:.4f}" for a, b in tab))
    print(f"  Best ensemble weight (classical) = {w}   Classical F1 = {m_lr['weighted_f1']:.4f}   Transformer F1 = {RISK_BEST_BY_ARCH[top_arch]['best_score']:.4f}   Ensemble F1 = {s:.4f}   vs first-review baseline: {s - BASELINE['risk_weighted_f1']:+.4f}")
    RISK_CANDS[f"Ensemble: LR + {top_arch}"] = dict(score=s, rank=5, proba=w * LR_VAL_PROBA + (1 - w) * RISK_BEST_BY_ARCH[top_arch]["val_proba"],
                                                     kind="ensemble", comp={"LR": w, top_arch: 1 - w}, blend_table=tab)
    if len(tr_archs) > 1:
        avg = np.mean([RISK_BEST_BY_ARCH[a]["val_proba"] for a in tr_archs], axis=0)
        RISK_CANDS["Ensemble: avg(" + ",".join(a.split("/")[-1].split("-")[0] for a in tr_archs) + ")"] = dict(
            score=sf(avg), rank=6, proba=avg, kind="ensemble", comp={a: 1 / len(tr_archs) for a in tr_archs})
        w2, s2, tab2 = blend_grid(LR_VAL_PROBA, avg, sf)
        print(f"Blend  LR(w) + avg(all Transformers)(1-w): best w={w2}  F1={s2:.4f}")
        RISK_CANDS["Ensemble: LR + avg(all Transformers)"] = dict(score=s2, rank=7, proba=w2 * LR_VAL_PROBA + (1 - w2) * avg, kind="ensemble",
                                                                  comp={"LR": w2, **{a: (1 - w2) / len(tr_archs) for a in tr_archs}}, blend_table=tab2)
for a in failed_archs: print(f"  (not available: {a} FAILED - excluded, not replaced by any number)")
for n, c in RISK_CANDS.items():
    if "pred" not in c: c["pred"] = c["proba"].argmax(1)
RISK_SEL, txt = select_best(RISK_CANDS, label="risk"); print("\n" + txt)
sel = RISK_CANDS[RISK_SEL]; RISK_VAL_PRED = sel["pred"]; RISK_VAL_PROBA = sel["proba"]
RISK_FINAL = risk_metrics(y_va_risk, RISK_VAL_PRED)
print(f"\nPrimary-split validation  Weighted F1={RISK_FINAL['weighted_f1']:.4f}  Macro F1={RISK_FINAL['macro_f1']:.4f}  Accuracy={RISK_FINAL['accuracy']:.4f}")
print(RISK_FINAL["per_class"].round(3).to_string())
RESULTS["risk"] = {"selected": RISK_SEL, "weighted_f1": RISK_FINAL["weighted_f1"], "macro_f1": RISK_FINAL["macro_f1"], "accuracy": RISK_FINAL["accuracy"],
                   "per_class": RISK_FINAL["per_class"].round(4).to_dict(), "components": sel["comp"],
                   "candidates": {n: {"score": float(c["score"]), "kind": c["kind"]} for n, c in RISK_CANDS.items()}}
cm = confusion_matrix(y_va_risk, RISK_VAL_PRED, labels=range(4))
fig, ax = plt.subplots(figsize=(5.8, 5)); im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_yticks(range(4)); ax.set_xticklabels(RISK_ORDER, rotation=20); ax.set_yticklabels(RISK_ORDER)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(f"Final risk model confusion matrix\n{RISK_SEL}", fontsize=10)
for i in range(4):
    for j in range(4): ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im); plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "risk_confusion_matrix.png"), dpi=130); plt.show()

### Robustness across seeds / group splits (risk)

In [ ]:
# The primary split decides model selection. Extra group splits test whether the gain is real or split luck.
def risk_on_split(seed, cfg):
    a, b = get_group_split(seed); ya = a.risk_level.map({r: i for i, r in enumerate(RISK_ORDER)}).values; yb = b.risk_level.map({r: i for i, r in enumerate(RISK_ORDER)}).values
    key = f"riskrob_{cfg['model']}_lr{cfg['lr']}_{cfg['loss']}_sp{seed}"
    r = cached(key, lambda: safe_run(key, lambda: train_transformer(cfg["model"], "multiclass", a.post_clean, ya, va_texts=b.post_clean, y_va=yb, lr=cfg["lr"], epochs=RISK_EPOCHS,
            micro_bs=cfg["mb"], accum=cfg["acc"], loss_type=cfg["loss"], max_len=MAX_LEN, seed=SEED, patience=RISK_PATIENCE, metric_fn=risk_metric_fn, num_labels=4)))
    return r, yb

rob_rows = {}
if RISK_BEST_BY_ARCH:
    top_cfg = RISK_BEST_BY_ARCH[max(RISK_BEST_BY_ARCH, key=lambda a: RISK_BEST_BY_ARCH[a]["best_score"])]
    rob_rows["LR (classical)"] = list(RESULTS["robustness"]["risk_LR"][:1])        # primary-split value first
    rob_rows[f"{top_cfg['model']}"] = [top_cfg["best_score"]]
    splits_used = [PRIMARY_SPLIT_SEED]
    for s in TRANSFORMER_ROBUST_SEEDS:
        print(f">> robustness split seed={s}: {top_cfg['model']}")
        r, yb = risk_on_split(s, top_cfg)
        if "failed" in r: continue
        rob_rows[top_cfg["model"]].append(r["best_score"]); splits_used.append(s)
        rob_rows["LR (classical)"].append(RESULTS["robustness"]["risk_LR"][CLASSICAL_ROBUST_SEEDS.index(s)])
    rb = pd.DataFrame({k: v for k, v in rob_rows.items()}, index=[f"split {s}" for s in splits_used[:len(next(iter(rob_rows.values())))]])
    rb.loc["MEAN"] = rb.mean(); rb.loc["STD"] = rb.iloc[:-1].std(ddof=0); print(rb.round(4).to_string())
    RESULTS["robustness"]["risk_table"] = rb.round(4).to_dict()
    print(f"\nTransformer - LR (mean over {len(splits_used)} splits): {rb.loc['MEAN', top_cfg['model']] - rb.loc['MEAN', 'LR (classical)']:+.4f}")
else: print("No successful Transformer run -> no Transformer robustness table.")
print("Classical 5-split robustness: LR mean=%.4f std=%.4f | SVM mean=%.4f std=%.4f" % (np.mean(rob["LR"]), np.std(rob["LR"]), np.mean(rob["SVM"]), np.std(rob["SVM"])))

### Paired user-level bootstrap — is the improvement bigger than validation noise?
The validation split has only ~30 users, so single-split differences can be noise (the first review already observed ±0.06 swings). We resample **users** with replacement (1000×) and report the 95 % interval of (final − baseline).

In [ ]:
def cluster_boot(fn_base, fn_final, groups, n=1000, seed=0):
    rng = np.random.RandomState(seed); groups = np.asarray(groups); users = np.unique(groups)
    idx_by = {u: np.where(groups == u)[0] for u in users}; d = []
    for _ in range(n):
        idx = np.concatenate([idx_by[u] for u in rng.choice(users, len(users), replace=True)])
        d.append(fn_final(idx) - fn_base(idx))
    d = np.array(d); return {"mean_diff": float(d.mean()), "ci_low": float(np.percentile(d, 2.5)), "ci_high": float(np.percentile(d, 97.5)), "p_better": float((d > 0).mean())}
GROUPS_VA = val_split.anon_user_id.values
bs = cluster_boot(lambda i: f1_score(y_va_risk[i], LR_VAL_PROBA.argmax(1)[i], average="weighted"), lambda i: f1_score(y_va_risk[i], RISK_VAL_PRED[i], average="weighted"), GROUPS_VA)
RESULTS["bootstrap"]["risk"] = bs
print(f"Risk  (final - classical LR): mean {bs['mean_diff']:+.4f} | 95% CI [{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}] | P(final better) = {bs['p_better']:.2f}")

## 15. Factor Classification  (multi-label, 24 factors)
## 16. Factor Threshold Optimisation
## 17. Factor Ensemble

### Algorithm 3 — Per-factor threshold optimisation
```
Algorithm   : TuneFactorThresholds
Input       : validation labels Y (n x 24), validation probabilities P (n x 24), grid G = {0.10, 0.15, ..., 0.90}
Output      : threshold vector tau (24)
Variables   : j (factor index), F1_j(t) (F1 of factor j at threshold t)
Steps:
  1. for j = 1 to 24 do
  2.     if sum(Y[:, j]) = 0 then tau[j] <- 0.5 ; continue       (no validation evidence for this factor)
  3.     tau[j] <- argmax over t in G of F1_j(t)                  (ties: the t closest to 0.5)
  4. return tau
```

In [ ]:
THR_GRID = np.round(np.arange(0.10, 0.901, 0.05), 2)
def _f1_vec(y, pred):
    tp = (y * pred).sum(0).astype(float); fp = ((1 - y) * pred).sum(0); fn = (y * (1 - pred)).sum(0)
    return np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
def tune_thresholds(y, p, grid=THR_GRID, default=0.5):
    th = np.full(y.shape[1], default)
    tab = np.stack([_f1_vec(y, (p >= t).astype(int)) for t in grid])          # (|grid|, 24)
    for j in range(y.shape[1]):
        if y[:, j].sum() == 0: continue
        best = tab[:, j].max(); cand = grid[np.isclose(tab[:, j], best)]; th[j] = cand[np.argmin(np.abs(cand - 0.5))]
    return th
def factor_metrics(y, pred):
    return {"macro_f1": f1_score(y, pred, average="macro", zero_division=0), "micro_f1": f1_score(y, pred, average="micro", zero_division=0),
            "weighted_f1": f1_score(y, pred, average="weighted", zero_division=0), "hamming_loss": hamming_loss(y, pred),
            "exact_match": float((y == pred).all(1).mean()), "per_factor": _f1_vec(y, pred)}
def macro_at_tuned(y, p): th = tune_thresholds(y, p); return factor_metrics(y, (p >= th).astype(int))["macro_f1"]
def crossfit_macro(y, p, groups, reps=10, seed=0):
    """Honest estimate: thresholds tuned on half of the validation USERS, scored on the other half (and vice versa)."""
    rng = np.random.RandomState(seed); users = np.unique(groups); out = []
    for _ in range(reps):
        rng.shuffle(users); A = np.isin(groups, users[: len(users) // 2]); pred = np.zeros_like(y)
        for tr_m, te_m in ((A, ~A), (~A, A)):
            th = tune_thresholds(y[tr_m], p[tr_m]); pred[te_m] = (p[te_m] >= th).astype(int)
        out.append(f1_score(y, pred, average="macro", zero_division=0))
    return float(np.mean(out)), float(np.std(out))

fac_new_thr_lr = tune_thresholds(Y_VA, LR_FAC_VAL_PROBA); lr_fac_metrics = factor_metrics(Y_VA, (LR_FAC_VAL_PROBA >= fac_new_thr_lr).astype(int))
print(f"Classical LR, thresholds on 0.10-0.90 grid: Macro F1 = {lr_fac_metrics['macro_f1']:.4f}  (first-review grid 0.20-0.60: {RESULTS['repro_factor']:.4f})")

FAC_RUNS = []
def factor_metric_fn(y, pr): return float(np.nanmean([average_precision_score(y[:, j], pr[:, j]) for j in range(y.shape[1]) if y[:, j].sum() > 0]))   # threshold-free -> early stopping does not pre-tune thresholds
for name, lrs, mb, acc in FACTOR_GRID:
    for lr in lrs:
        key = f"factor_{name}_lr{lr}_wbce_s{SEED}_sp{PRIMARY_SPLIT_SEED}"; print(f"\n>> FACTORS {name} lr={lr} loss=wbce")
        r = cached(key, lambda: safe_run(key, lambda: train_transformer(name, "multilabel", train_split.post_clean, Y_TR, va_texts=val_split.post_clean, y_va=Y_VA, lr=lr,
              epochs=FACTOR_EPOCHS, micro_bs=mb, accum=acc, loss_type="wbce", max_len=MAX_LEN, seed=SEED, patience=FACTOR_PATIENCE, metric_fn=factor_metric_fn,
              num_labels=N_FACTORS, head_lr_mult=FACTOR_HEAD_LR_MULT)))
        r.setdefault("model", name); r["mb"], r["acc"] = mb, acc; FAC_RUNS.append(r)
        if "failed" not in r: r["macro_tuned"] = macro_at_tuned(Y_VA, r["val_proba"]); print(f"   done: val mAP={r['best_score']:.4f}  Macro-F1 (tuned thr)={r['macro_tuned']:.4f} @epoch {r['best_epoch']}  {r['seconds']:.0f}s")
fok = [r for r in FAC_RUNS if "failed" not in r]
if fok:       # loss ablation on the best architecture
    bf = max(fok, key=lambda r: r["macro_tuned"])
    for lt in ["bce", "focal"]:
        key = f"factor_{bf['model']}_lr{bf['lr']}_{lt}_s{SEED}_sp{PRIMARY_SPLIT_SEED}"; print(f"\n>> FACTORS loss ablation {lt}")
        r = cached(key, lambda: safe_run(key, lambda: train_transformer(bf["model"], "multilabel", train_split.post_clean, Y_TR, va_texts=val_split.post_clean, y_va=Y_VA, lr=bf["lr"],
              epochs=FACTOR_EPOCHS, micro_bs=bf["mb"], accum=bf["acc"], loss_type=lt, max_len=MAX_LEN, seed=SEED, patience=FACTOR_PATIENCE, metric_fn=factor_metric_fn,
              num_labels=N_FACTORS, head_lr_mult=FACTOR_HEAD_LR_MULT)))
        r.setdefault("model", bf["model"]); r["mb"], r["acc"] = bf["mb"], bf["acc"]; FAC_RUNS.append(r)
        if "failed" not in r: r["macro_tuned"] = macro_at_tuned(Y_VA, r["val_proba"]); print(f"   {lt}: Macro-F1 (tuned thr)={r['macro_tuned']:.4f}")
fok = [r for r in FAC_RUNS if "failed" not in r]
FAC_BEST_BY_ARCH = {}
for r in fok:
    if r["model"] not in FAC_BEST_BY_ARCH or r["macro_tuned"] > FAC_BEST_BY_ARCH[r["model"]]["macro_tuned"]: FAC_BEST_BY_ARCH[r["model"]] = r
print("\n"); print(pd.DataFrame([{"Model": r["model"], "LR": r.get("lr"), "Loss": r.get("loss"), "Batch": r.get("batch"), "Val mAP": r.get("best_score"), "Val Macro F1 (tuned thr)": r.get("macro_tuned"),
      "Best epoch": r.get("best_epoch"), "Time(s)": r.get("seconds"), "Status": r.get("failed", "ok")} for r in FAC_RUNS]).round(4).to_string(index=False))
assert fok or SMOKE, "No Transformer factor run succeeded - do not report Transformers as executed."

In [ ]:
FAC_CANDS = {"TF-IDF + OvR Logistic Regression": dict(score=lr_fac_metrics["macro_f1"], rank=0, proba=LR_FAC_VAL_PROBA, kind="classical", comp={"LR": 1.0})}
for arch, r in FAC_BEST_BY_ARCH.items():
    FAC_CANDS[arch] = dict(score=r["macro_tuned"], rank=ARCH_RANK.get(arch, 2), proba=r["val_proba"], kind="transformer", comp={arch: 1.0}, run=r)
sfm = lambda p: macro_at_tuned(Y_VA, p)          # every candidate is scored AFTER its own per-factor threshold optimisation
fa = list(FAC_BEST_BY_ARCH)
if fa:
    top = max(fa, key=lambda a: FAC_BEST_BY_ARCH[a]["macro_tuned"])
    w, s, tab = blend_grid(LR_FAC_VAL_PROBA, FAC_BEST_BY_ARCH[top]["val_proba"], sfm)
    print(f"Blend LR(w) + {top}(1-w), Macro F1 after threshold tuning: " + "  ".join(f"{a:.1f}:{b:.4f}" for a, b in tab))
    print(f"  best w={w}  classical={lr_fac_metrics['macro_f1']:.4f}  transformer={FAC_BEST_BY_ARCH[top]['macro_tuned']:.4f}  ensemble={s:.4f}")
    FAC_CANDS[f"Ensemble: LR + {top}"] = dict(score=s, rank=5, proba=w * LR_FAC_VAL_PROBA + (1 - w) * FAC_BEST_BY_ARCH[top]["val_proba"], kind="ensemble", comp={"LR": w, top: 1 - w})
    if len(fa) > 1:
        avg = np.mean([FAC_BEST_BY_ARCH[a]["val_proba"] for a in fa], axis=0)
        FAC_CANDS["Ensemble: avg(Transformers)"] = dict(score=sfm(avg), rank=6, proba=avg, kind="ensemble", comp={a: 1 / len(fa) for a in fa})
        w2, s2, _ = blend_grid(LR_FAC_VAL_PROBA, avg, sfm)
        FAC_CANDS["Ensemble: LR + avg(Transformers)"] = dict(score=s2, rank=7, proba=w2 * LR_FAC_VAL_PROBA + (1 - w2) * avg, kind="ensemble", comp={"LR": w2, **{a: (1 - w2) / len(fa) for a in fa}})
FAC_SEL, txt = select_best(FAC_CANDS, label="factors"); print("\n" + txt)
fsel = FAC_CANDS[FAC_SEL]; FAC_VAL_PROBA = fsel["proba"]; FAC_THR = tune_thresholds(Y_VA, FAC_VAL_PROBA)
FAC_VAL_PRED = (FAC_VAL_PROBA >= FAC_THR).astype(int); FAC_FINAL = factor_metrics(Y_VA, FAC_VAL_PRED)
print(f"\nFINAL factor model on primary split: Macro F1={FAC_FINAL['macro_f1']:.4f} Micro F1={FAC_FINAL['micro_f1']:.4f} Weighted F1={FAC_FINAL['weighted_f1']:.4f} "
      f"Hamming loss={FAC_FINAL['hamming_loss']:.4f} Exact match={FAC_FINAL['exact_match']:.4f}")
cf_f, cf_s = crossfit_macro(Y_VA, FAC_VAL_PROBA, GROUPS_VA); cf_b, cf_bs = crossfit_macro(Y_VA, LR_FAC_VAL_PROBA, GROUPS_VA)
print(f"Cross-fitted (thresholds tuned on half the val users, scored on the other half): final={cf_f:.4f}±{cf_s:.4f} | classical LR={cf_b:.4f}±{cf_bs:.4f}")
print("  -> the gap between the tuned-on-val number and this one is the threshold-tuning optimism; it affects the first review and the final review alike.")
fac_tab = pd.DataFrame({"factor": OFFICIAL_FACTORS, "train_support": Y_TR.sum(0), "val_support": Y_VA.sum(0), "threshold": FAC_THR,
                        "F1 first-review model": _f1_vec(Y_VA, (LR_FAC_VAL_PROBA >= th_old).astype(int)).round(3), "F1 final": FAC_FINAL["per_factor"].round(3)}).sort_values("train_support", ascending=False)
print("\n"); print(fac_tab.to_string(index=False))
RESULTS["factor"] = {"selected": FAC_SEL, **{k: float(v) for k, v in FAC_FINAL.items() if k != "per_factor"}, "crossfit_macro": cf_f, "crossfit_macro_baseline_model": cf_b,
                     "thresholds": dict(zip(OFFICIAL_FACTORS, map(float, FAC_THR))), "components": fsel["comp"],
                     "candidates": {n: {"score": float(c["score"]), "kind": c["kind"]} for n, c in FAC_CANDS.items()}}
bsf = cluster_boot(lambda i: f1_score(Y_VA[i], (LR_FAC_VAL_PROBA >= th_old).astype(int)[i], average="macro", zero_division=0), lambda i: f1_score(Y_VA[i], FAC_VAL_PRED[i], average="macro", zero_division=0), GROUPS_VA)
RESULTS["bootstrap"]["factor"] = bsf; print(f"\nFactors (final - first-review model) user-bootstrap: mean {bsf['mean_diff']:+.4f} | 95% CI [{bsf['ci_low']:+.4f}, {bsf['ci_high']:+.4f}] | P(better)={bsf['p_better']:.2f}")
fig, ax = plt.subplots(figsize=(10, 6)); ft = fac_tab.iloc[::-1]; y = np.arange(len(ft))
ax.barh(y - 0.2, ft["F1 first-review model"], 0.4, label="first review", color="#9aa5b1"); ax.barh(y + 0.2, ft["F1 final"], 0.4, label="final", color="#2a6f97")
ax.set_yticks(y); ax.set_yticklabels(ft.factor, fontsize=8); ax.set_xlabel("F1 (primary validation split)"); ax.legend(); ax.set_title("Per-factor F1: first review vs final"); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "factor_f1_comparison.png"), dpi=130); plt.show()

## 14. Evidence Extraction
**Hard rule:** every returned span is a slice `post[start:end]` of the original post (character offsets), so it is a verbatim substring by construction.

**First-review extractor (reproduced):** sentences + clauses → TF-IDF LR → probability ≥ 0.7. It ignored the risk class.
**Final extractor:** (1) predicted risk gates the post — gold evidence is the string `none` for ~95 % of *Indicator* posts, so predicting a span there can only lose;
(2) each candidate span is scored by a Transformer cross-encoder that sees the predicted risk (and optionally the post); (3) hybrid blend with TF-IDF LR;
(4) decoding parameters (threshold, max spans, shortest-span pruning, top-1 fallback) tuned on validation.

### Algorithm 4 — Risk-aware verbatim evidence extraction
```
Algorithm   : ExtractEvidence
Input       : post T, predicted risk r, scorer S, decoding parameters (thr, K, prune, fallback)
Output      : list of evidence spans, each an exact substring of T
Variables   : C (candidate offsets), p (scores), kept (selected spans)
Steps:
  1. if r = Indicator then return empty list
  2. C <- sentences of T (split on newline and . ! ?) together with their clauses (split on ; : , and conjunctions); store (start, end) offsets
  3. for each c in C do p[c] <- S(r, T[c.start:c.end])
  4. P <- { c in C : p[c] >= thr }
  5. if P is empty and fallback then P <- { argmax p }
  6. if prune then sort P by token length ascending and drop any span that contains an already kept shorter span
  7. keep the K highest-scoring spans of P
  8. return { T[c.start:c.end] : c in kept }
```

In [ ]:
# ---------------- first-review evidence code (verbatim logic, kept for the reproduction) ----------------
def get_spans(s): return [x.strip() for x in str(s).split(";") if x.strip()]
def tokenize(t): return t.split()
def spans_match(pred, gt):
    p, g = pred.lower().strip(), gt.lower().strip()
    if not p or not g: return False
    if not ((g in p) or (p in g)): return False
    gl = len(tokenize(g)); return gl > 0 and len(tokenize(p)) <= 3 * gl
def match_spans(preds, gts):
    un = list(gts); m = 0
    for p in preds:
        for i, g in enumerate(un):
            if spans_match(p, g): m += 1; un.pop(i); break
    return m
def phrase_f1_for_post(preds, gts, match=match_spans):
    if len(gts) == 0: return (1.0, 1.0, 1.0) if len(preds) == 0 else (0.0, 0.0, 0.0)
    if len(preds) == 0: return (0.0, 0.0, 0.0)
    m = match(preds, gts); p, r = m / len(preds), m / len(gts); return p, r, (0.0 if p + r == 0 else 2 * p * r / (p + r))

def _sent_split(text):
    out = []
    for line in str(text).split("\n"):
        line = line.strip()
        if line: out.extend([p.strip() for p in re.split(r"(?<=[.!?])\s+", line) if p.strip()])
    return out
CLAUSE_RE = re.compile(r"[;:,]|\s+(?:and|but|or|because|so|since|although|though|when|while)\s+", flags=re.IGNORECASE)
def _clauses(s): return [p.strip(" .!?\"'") for p in CLAUSE_RE.split(s) if p.strip(" .!?\"'")]
def generate_candidates_baseline(text):
    c = []
    for s in _sent_split(text):
        c.append(s); c.extend(x for x in _clauses(s) if x != s)
    seen, out = set(), []
    for x in c:
        if x.lower() not in seen: seen.add(x.lower()); out.append(x)
    return out
def predict_evidence_baseline(text, vec, clf, thr):
    cands = generate_candidates_baseline(text)
    if not cands: return []
    pr = clf.predict_proba(vec.transform([c.lower() for c in cands]))[:, 1]
    pos = sorted([(c, p) for c, p in zip(cands, pr) if p >= thr], key=lambda x: len(tokenize(x[0]))); kept = []
    for c, _ in pos:
        if any(c.lower() in k.lower() for k in kept): continue
        kept.append(c)
    return kept

val_ev = val_split.dropna(subset=["evidence for suicide risk level"]).copy()           # same evaluation population as the first review
tr_ev = train_split.dropna(subset=["evidence for suicide risk level"])
def gold_spans(row): return [s for s in get_spans(row["evidence for suicide risk level"]) if s.lower() != "none"]
txts, labs = [], []
for _, row in tr_ev.iterrows():
    g = get_spans(row["evidence for suicide risk level"]); cs = generate_candidates_baseline(row["post_clean"]); txts += cs
    labs += [1 if any(spans_match(c, x) for x in g if x.lower() != "none") else 0 for c in cs]
ev_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000); ev_X = ev_vec.fit_transform([t.lower() for t in txts])
ev_lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED).fit(ev_X, labs)
def eval_baseline_extractor(rows, pred_fn, match=match_spans):
    f = [phrase_f1_for_post(pred_fn(r), gold_spans(r), match)[2] for _, r in rows.iterrows()]; return float(np.mean(f)), np.array(f)
E0, E0_arr = eval_baseline_extractor(val_ev, lambda r: predict_evidence_baseline(r["post_clean"], ev_vec, ev_lr, 0.7))
print(f"First-review evidence extractor reproduced: Phrase F1 = {E0:.4f}   (frozen first-review value: {BASELINE['evidence_phrase_f1']:.4f})")
RESULTS["repro_evidence"] = E0

In [ ]:
# ---------------- offset-based candidate generation: guarantees verbatim substrings of the RAW post ----------------
def _trim(text, a, b, chars):
    while a < b and text[a] in chars: a += 1
    while b > a and text[b - 1] in chars: b -= 1
    return a, b
WS = " \t\r\n"; PUNCT = WS + ".!?\"'"
def candidate_spans(post):
    post = "" if pd.isna(post) else str(post); out = []; seen = set()
    def add(a, b):
        if b <= a: return
        key = re.sub(r"\s+", " ", post[a:b]).strip().lower()
        if key and key not in seen: seen.add(key); out.append((a, b))
    for lm in re.finditer(r"[^\n]+", post):
        ls, line = lm.start(), lm.group(); last = 0; sents = []
        for m in re.finditer(r"(?<=[.!?])\s+", line): sents.append((last, m.start())); last = m.end()
        sents.append((last, len(line)))
        for ss, se in sents:
            a, b = _trim(post, ls + ss, ls + se, WS)
            if b <= a: continue
            add(a, b); sent = post[a:b]; last = 0; segs = []
            for m in CLAUSE_RE.finditer(sent): segs.append((last, m.start())); last = m.end()
            segs.append((last, len(sent)))
            for cs, ce in segs:
                a2, b2 = _trim(post, a + cs, a + ce, PUNCT)
                if (a2, b2) != (a, b): add(a2, b2)
    return out
norm = lambda s: re.sub(r"\s+", " ", s).strip()
def to_raw(post, span):
    """map a whitespace-normalised span back to the exact raw slice of the post (None if not found)"""
    m = re.search(r"\s+".join(map(re.escape, span.split())), str(post)); return m.group(0) if m else None

def build_cands(df):
    M = {}
    for _, r in df.iterrows():
        post = "" if pd.isna(r["post"]) else str(r["post"]); sp = candidate_spans(post)
        M[r["row_id"]] = {"post": post, "spans": sp, "raw": [post[a:b] for a, b in sp], "norm": [norm(post[a:b]) for a, b in sp]}
    return M
CAND_TR, CAND_VA = build_cands(train_split), build_cands(val_split)
for M in (CAND_TR, CAND_VA):
    for v in M.values(): assert all(v["post"][a:b] == t for (a, b), t in zip(v["spans"], v["raw"]))     # verbatim by construction
print("avg candidates / post:", np.mean([len(v["raw"]) for v in CAND_VA.values()]).round(1))

# oracle ceiling of the candidate generator (diagnostic: best achievable if the scorer were perfect)
def oracle_f1(row, M):
    g = gold_spans(row); c = M[row["row_id"]]["norm"]
    if not g: return 1.0
    hit = [x for x in c if any(spans_match(x, y) for y in g)]
    return phrase_f1_for_post(hit[:len(g)] if hit else [], g)[2]
ORACLE = float(np.mean([oracle_f1(r, CAND_VA) for _, r in val_ev.iterrows()]))
print(f"Candidate-generator oracle ceiling (Phrase F1): {ORACLE:.4f}")

# strict sanity metric: token-level containment (guards against the char-level containment loophole of the local approximation)
def _tk(s): return re.findall(r"\w+|[^\w\s]", s.lower())
def _sub(a, b): return any(b[i:i + len(a)] == a for i in range(len(b) - len(a) + 1)) if a else False
def spans_match_strict(p, g):
    tp, tg = _tk(p), _tk(g)
    return bool(tp and tg and (_sub(tp, tg) or _sub(tg, tp)) and len(tp) <= 3 * len(tg))
def match_strict(preds, gts):
    un = list(gts); m = 0
    for p in preds:
        for i, g in enumerate(un):
            if spans_match_strict(p, g): m += 1; un.pop(i); break
    return m

In [ ]:
# ---------------- candidate scorers ----------------
RISK_LABEL_OF = lambda i: RISK_ORDER[int(i)]
pred_risk_va = dict(zip(val_split.row_id, RISK_VAL_PRED))        # PREDICTED risk of the selected risk model (honest pipeline evaluation)
gold_risk_va = dict(zip(val_split.row_id, y_va_risk))

def lr_scores(M, ids): return {i: ev_lr.predict_proba(ev_vec.transform([t.lower() for t in M[i]["norm"]]))[:, 1] if M[i]["norm"] else np.zeros(0) for i in ids}

def build_pairs(M, df, use_ctx, risk_of):
    A, B, Y, owner = [], [], [], []
    for _, r in df.iterrows():
        rid = r["row_id"]; g = gold_spans(r) if not pd.isna(r.get("evidence for suicide risk level", np.nan)) else []
        for t, raw in zip(M[rid]["norm"], M[rid]["raw"]):
            A.append(f"{RISK_ORDER[risk_of(rid)]}: {t}"); B.append(norm(M[rid]["post"]) if use_ctx else None)
            Y.append(int(any(spans_match(t, x) for x in g))); owner.append(rid)
    return A, (B if use_ctx else None), np.array(Y), owner

# training pairs: non-Indicator posts only (Indicator posts carry no evidence); GOLD risk text is used at training time
gold_risk_tr = dict(zip(train_split.row_id, train_split.risk_level.map({r: i for i, r in enumerate(RISK_ORDER)})))
tr_nonind = train_split[(train_split.risk_level != "Indicator") & train_split["evidence for suicide risk level"].notna()]
va_pred_nonind = val_split[val_split.row_id.map(lambda i: RISK_ORDER[int(pred_risk_va[i])] != "Indicator")]
ev_metric = lambda y, pr: average_precision_score(y, pr[:, 1])

EV_RUNS = {}
ev_arch = (max(RISK_BEST_BY_ARCH, key=lambda a: RISK_BEST_BY_ARCH[a]["best_score"]) if RISK_BEST_BY_ARCH else (RISK_GRID[0][0]))
ev_mb, ev_acc = [(m, a) for n, _, m, a in RISK_GRID if n == ev_arch][0]
print("Evidence scorer backbone:", ev_arch)
for use_ctx in (False, True):
    A, B, Y, _ = build_pairs(CAND_TR, tr_nonind, use_ctx, lambda i: gold_risk_tr[i])
    Av, Bv, Yv, own_v = build_pairs(CAND_VA, va_pred_nonind, use_ctx, lambda i: int(pred_risk_va[i]))   # validation labels (gold evidence) are used for early stopping only
    key = f"evid_{ev_arch}_ctx{int(use_ctx)}_s{SEED}"; print(f"\n>> EVIDENCE scorer {ev_arch} context={use_ctx}  train pairs={len(A)} (pos rate {Y.mean():.3f})")
    ml = MAX_LEN if use_ctx else min(MAX_LEN, 96)
    r = cached(key, lambda: safe_run(key, lambda: train_transformer(ev_arch, "multiclass", A, Y, va_texts=Av, y_va=Yv, tr_b=B, va_b=Bv, lr=EVID_LR, epochs=EVID_EPOCHS, micro_bs=ev_mb if use_ctx else 32,
             accum=ev_acc if use_ctx else 1, loss_type="wce", max_len=ml, seed=SEED, patience=EVID_PATIENCE, metric_fn=ev_metric, num_labels=2)))
    if "failed" not in r:
        r["owner"] = own_v; r["use_ctx"] = use_ctx; r["max_len_used"] = ml; print(f"   done: candidate-level AP={r['best_score']:.4f} epoch={r['best_epoch']} {r['seconds']:.0f}s")
    EV_RUNS[use_ctx] = r

In [ ]:
# ---------------- decoding + evaluation of every evidence system ----------------
def tf_scores_from_run(run, M):
    d = {}
    for rid, p in zip(run["owner"], run["val_proba"][:, 1]): d.setdefault(rid, []).append(p)
    return {k: np.array(v) for k, v in d.items()}

def decode(M_i, probs, thr, K, prune, fallback):
    n = len(probs)
    if n == 0: return []
    idx = [i for i in range(n) if probs[i] >= thr]
    if not idx and fallback: idx = [int(np.argmax(probs))]
    if prune:
        idx = sorted(idx, key=lambda i: len(M_i["norm"][i].split())); kept = []
        for i in idx:
            if any(M_i["norm"][k].lower() in M_i["norm"][i].lower() for k in kept): continue       # drop longer spans that contain a shorter kept one
            kept.append(i)
        idx = kept
    idx = sorted(idx, key=lambda i: -probs[i])[:K]
    return [M_i["raw"][i] for i in idx]

def eval_system(score_map, params, rows=val_ev, risk_pred=pred_risk_va, M=CAND_VA, strict=False):
    f = []
    for _, r in rows.iterrows():
        rid = r["row_id"]
        preds = [] if RISK_ORDER[int(risk_pred[rid])] == "Indicator" else decode(M[rid], score_map.get(rid, np.zeros(len(M[rid]["raw"]))), *params)
        f.append(phrase_f1_for_post([norm(p) for p in preds], gold_spans(r), match_strict if strict else match_spans)[2])
    return float(np.mean(f)), np.array(f)

PARAM_GRID = [(t, K, pr, fb) for t in np.round(np.arange(0.1, 0.91, 0.1), 1) for K in (1, 2, 3, 99) for pr in (False, True) for fb in (False, True)]
def tune_decode(score_map):
    best = (-1, None)
    for p in PARAM_GRID:
        s, _ = eval_system(score_map, p)
        if s > best[0] + 1e-9: best = (s, p)
    return best
ids_va = list(val_split.row_id); LRS = lr_scores(CAND_VA, ids_va)

# E1: first-review LR scorer + predicted-risk gate, first-review decoding (thr .7, no fallback, no cap)
E1, E1_arr = eval_baseline_extractor(val_ev, lambda r: [] if RISK_ORDER[int(pred_risk_va[r["row_id"]])] == "Indicator" else predict_evidence_baseline(r["post_clean"], ev_vec, ev_lr, 0.7))
EV_SYS = {"First-review extractor (reproduced, no risk gate)": dict(score=E0, rank=0, arr=E0_arr, spec=None),
          "TF-IDF LR + predicted-risk gate": dict(score=E1, rank=1, arr=E1_arr, spec=None)}
s_lr, p_lr = tune_decode(LRS); _, a_lr = eval_system(LRS, p_lr)
EV_SYS["TF-IDF LR + gate + tuned decoding"] = dict(score=s_lr, rank=2, arr=a_lr, spec={"kind": "lr", "params": p_lr})
tf_maps = {}
for use_ctx, run in EV_RUNS.items():
    if "failed" in run: print(f"   (evidence Transformer ctx={use_ctx} FAILED - excluded)"); continue
    tf_maps[use_ctx] = tf_scores_from_run(run, CAND_VA)
    sm = {i: tf_maps[use_ctx].get(i, np.zeros(len(CAND_VA[i]["raw"]))) for i in ids_va}; s, p = tune_decode(sm); _, arr = eval_system(sm, p)
    EV_SYS[f"{ev_arch.split('/')[-1]} cross-encoder{' + post context' if use_ctx else ''} + gate"] = dict(score=s, rank=3 + int(use_ctx), arr=arr, spec={"kind": "tf", "ctx": use_ctx, "params": p})
if tf_maps:
    bc = max(tf_maps, key=lambda c: EV_SYS[[k for k in EV_SYS if (('post context' in k) == c) and 'cross-encoder' in k][0]]["score"])
    for w in (0.25, 0.5, 0.75):
        sm = {i: w * tf_maps[bc].get(i, np.zeros(len(CAND_VA[i]["raw"]))) + (1 - w) * LRS[i] for i in ids_va}; s, p = tune_decode(sm); _, arr = eval_system(sm, p)
        EV_SYS[f"Hybrid: {w:.2f}*Transformer + {1-w:.2f}*TF-IDF LR + gate"] = dict(score=s, rank=6, arr=arr, spec={"kind": "hybrid", "ctx": bc, "w": w, "params": p})
EV_SEL, txt = select_best(EV_SYS, label="evidence"); print(txt)
print("\nEvidence systems (primary validation split, rows with a gold evidence field):")
tab = pd.DataFrame([{"System": k, "Phrase F1": v["score"]} for k, v in EV_SYS.items()]); print(tab.round(4).to_string(index=False))
print(f"Candidate oracle ceiling: {ORACLE:.4f}")
EV_FINAL = EV_SYS[EV_SEL]
# strict-metric sanity check for baseline vs final
def strict_for(sys_name):
    sp = EV_SYS[sys_name]["spec"]
    if sp is None: return None
    if sp["kind"] == "lr": sm = LRS
    elif sp["kind"] == "tf": sm = {i: tf_maps[sp["ctx"]].get(i, np.zeros(len(CAND_VA[i]["raw"]))) for i in ids_va}
    else: sm = {i: sp["w"] * tf_maps[sp["ctx"]].get(i, np.zeros(len(CAND_VA[i]["raw"]))) + (1 - sp["w"]) * LRS[i] for i in ids_va}
    return eval_system(sm, sp["params"], strict=True)[0], sm
base_strict, _ = eval_baseline_extractor(val_ev, lambda r: predict_evidence_baseline(r["post_clean"], ev_vec, ev_lr, 0.7), match_strict)
st = strict_for(EV_SEL)
if st: print(f"Strict token-level metric (sanity): first-review extractor {base_strict:.4f} -> selected final {st[0]:.4f}")
EV_SCOREMAP_VAL = st[1] if st else None
bse = cluster_boot(lambda i: E0_arr[i].mean(), lambda i: EV_FINAL["arr"][i].mean(), val_ev.anon_user_id.values)
RESULTS["bootstrap"]["evidence"] = bse
print(f"Evidence (final - first-review extractor) user-bootstrap: mean {bse['mean_diff']:+.4f} | 95% CI [{bse['ci_low']:+.4f}, {bse['ci_high']:+.4f}] | P(better)={bse['p_better']:.2f}")
RESULTS["evidence"] = {"selected": EV_SEL, "phrase_f1": EV_FINAL["score"], "spec": str(EV_FINAL["spec"]),
                       "oracle_ceiling": ORACLE, "strict_final": st[0] if st else None, "strict_first_review": base_strict,
                       "systems": {k: float(v["score"]) for k, v in EV_SYS.items()}}

## 18. Model Comparison  (every executed model, one table)

In [ ]:
rows = []
for n, c in RISK_CANDS.items(): rows.append(("Risk", n, "Weighted F1", c["score"], n == RISK_SEL))
for a in failed_archs: rows.append(("Risk", a, "Weighted F1", np.nan, False))
for n, c in FAC_CANDS.items(): rows.append(("Factors", n, "Macro F1", c["score"], n == FAC_SEL))
for n, c in EV_SYS.items(): rows.append(("Evidence", n, "Phrase F1", c["score"], n == EV_SEL))
COMPARISON = pd.DataFrame(rows, columns=["Task", "Model", "Metric", "Score", "Selected?"])
COMPARISON["Selected?"] = COMPARISON["Selected?"].map({True: "YES  <-", False: ""}); COMPARISON["Score"] = COMPARISON["Score"].round(4)
print(COMPARISON.to_string(index=False)); COMPARISON.to_csv(os.path.join(OUT_DIR, "model_comparison.csv"), index=False)

## 19. FIRST REVIEW vs FINAL REVIEW

In [ ]:
FINAL = {"risk_weighted_f1": float(RISK_FINAL["weighted_f1"]), "evidence_phrase_f1": float(EV_FINAL["score"]), "factor_macro_f1": float(FAC_FINAL["macro_f1"])}
LABELS = {"risk_weighted_f1": "Risk Weighted F1", "evidence_phrase_f1": "Evidence Phrase F1", "factor_macro_f1": "Factor Macro F1"}
cmp_rows = []
for k in BASELINE:
    b, f = BASELINE[k], FINAL[k]; cmp_rows.append({"Metric": LABELS[k], "First Review": b, "Final Review": f, "Absolute Improvement": f - b, "Relative Improvement %": (f - b) / b * 100})
CMP = pd.DataFrame(cmp_rows); print(CMP.round(4).to_string(index=False))
RESULTS["final"] = FINAL; RESULTS["comparison"] = CMP.round(6).to_dict("records")

fig, ax = plt.subplots(1, 2, figsize=(14, 5.2)); x = np.arange(3); w = 0.36
b1 = ax[0].bar(x - w / 2, [BASELINE[k] for k in BASELINE], w, label="First Review", color="#9aa5b1"); b2 = ax[0].bar(x + w / 2, [FINAL[k] for k in BASELINE], w, label="Final Review", color="#2a6f97")
for bars in (b1, b2):
    for b in bars: ax[0].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f"{b.get_height():.4f}", ha="center", fontsize=10, fontweight="bold")
ax[0].set_xticks(x); ax[0].set_xticklabels([LABELS[k] for k in BASELINE]); ax[0].set_ylim(0, max(max(FINAL.values()), 0.7) + 0.12); ax[0].set_ylabel("Score"); ax[0].legend(); ax[0].set_title("FIRST REVIEW vs FINAL REVIEW  (primary validation split)", fontweight="bold")
imp = CMP["Relative Improvement %"].values; colors = ["#2a9d8f" if v >= 0 else "#c1121f" for v in imp]; bb = ax[1].bar([LABELS[k] for k in BASELINE], imp, color=colors)
for b, v in zip(bb, imp): ax[1].text(b.get_x() + b.get_width() / 2, v + (0.6 if v >= 0 else -2.2), f"{v:+.2f}%", ha="center", fontsize=11, fontweight="bold")
ax[1].axhline(0, color="black", lw=0.8); ax[1].set_ylabel("Relative improvement (%)"); ax[1].set_title("Improvement over first review (%)", fontweight="bold")
for a in ax: a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "first_vs_final.png"), dpi=150); plt.show()

# pipeline diagram
fig, ax = plt.subplots(figsize=(14, 2.6)); ax.axis("off")
steps = ["FIRST REVIEW\nbaseline\n0.6468 / 0.5599 / 0.4419", "MODEL\nIMPROVEMENT\n(Transformers, loss,\nthresholds, gate)", "FINAL REVIEW\nperformance\n(measured)", "IMPROVEMENT %", "FINAL MODEL\nSELECTION", "FINAL\nPREDICTIONS\n(CSV)"]
for i, s in enumerate(steps):
    ax.text(i * 2.5, 0.5, s, ha="center", va="center", fontsize=9, bbox=dict(boxstyle="round,pad=0.5", fc="#e8f1f8" if i not in (0, 5) else "#f2e8cf", ec="#2a6f97"))
    if i < len(steps) - 1: ax.annotate("", xy=(i * 2.5 + 1.55, 0.5), xytext=(i * 2.5 + 0.95, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))
ax.set_xlim(-1.2, 13.7); ax.set_ylim(0, 1); plt.savefig(os.path.join(OUT_DIR, "pipeline_diagram.png"), dpi=140, bbox_inches="tight"); plt.show()

### Evidence-score attribution (where did the evidence gain come from?)

In [ ]:
attr = pd.DataFrame([{"Step": k, "Phrase F1": v["score"]} for k, v in EV_SYS.items()]).round(4)
print(attr.to_string(index=False)); print("\nRead this table honestly: the jump from row 1 to row 2 is the risk gate alone (no better scorer). Anything above row 2/3 is the Transformer's contribution.")
RESULTS["evidence"]["attribution"] = attr.to_dict("records")

## 23 (analysis). Error Analysis — final models on the primary validation split

In [ ]:
exc = lambda t, n=130: (t[:n] + "...") if len(t) > n else t
va = val_split.copy(); va["pred"] = [RISK_ORDER[i] for i in RISK_VAL_PRED]; va["ok"] = va.pred == va.risk_level
print("RISK - correct examples"); print(va[va.ok].sample(5, random_state=1)[["risk_level", "pred", "post_clean"]].assign(post_clean=lambda d: d.post_clean.map(exc)).rename(columns={"risk_level": "Actual", "pred": "Predicted", "post_clean": "Post excerpt"}).to_string(index=False))
print("\nRISK - incorrect examples"); print(va[~va.ok].sample(min(8, (~va.ok).sum()), random_state=1)[["risk_level", "pred", "post_clean"]].assign(post_clean=lambda d: d.post_clean.map(exc)).rename(columns={"risk_level": "Actual", "pred": "Predicted", "post_clean": "Post excerpt"}).to_string(index=False))
pairs = va[~va.ok].groupby(["risk_level", "pred"]).size().sort_values(ascending=False).head(5); print("\nMost common confusions (actual -> predicted):"); print(pairs.to_string())
adj = np.mean([abs(RISK_ORDER.index(a) - RISK_ORDER.index(p)) == 1 for a, p in zip(va[~va.ok].risk_level, va[~va.ok].pred)]) if (~va.ok).any() else 0
print(f"Share of errors that are between ADJACENT severity levels: {adj:.2f}")

print("\nFACTORS - actual vs predicted (8 random validation posts)")
for i in np.random.RandomState(3).choice(len(val_split), 8, replace=False):
    act = {OFFICIAL_FACTORS[j] for j in np.where(Y_VA[i])[0]}; prd = {OFFICIAL_FACTORS[j] for j in np.where(FAC_VAL_PRED[i])[0]}
    print(f"  #{i}: ACTUAL={sorted(act)}\n        PRED  ={sorted(prd)}\n        missed={sorted(act - prd)}  extra={sorted(prd - act)}")
fp = (FAC_VAL_PRED * (1 - Y_VA)).sum(0); fn = ((1 - FAC_VAL_PRED) * Y_VA).sum(0)
print("\nTop false-positive factors:", [(OFFICIAL_FACTORS[j], int(fp[j])) for j in np.argsort(-fp)[:4]]); print("Top false-negative factors:", [(OFFICIAL_FACTORS[j], int(fn[j])) for j in np.argsort(-fn)[:4]])
print("Rare factors (train support < 30) with F1 = 0 on validation:", [OFFICIAL_FACTORS[j] for j in range(N_FACTORS) if Y_TR[:, j].sum() < 30 and FAC_FINAL["per_factor"][j] == 0])

print("\nEVIDENCE - gold vs predicted (final system)")
def final_val_evidence(r):
    rid = r["row_id"]
    if RISK_ORDER[int(pred_risk_va[rid])] == "Indicator": return []
    if EV_SYS[EV_SEL]["spec"] is None: return predict_evidence_baseline(r["post_clean"], ev_vec, ev_lr, 0.7)
    return decode(CAND_VA[rid], EV_SCOREMAP_VAL[rid], *EV_SYS[EV_SEL]["spec"]["params"])
ev_rows = val_ev.assign(pred=[final_val_evidence(r) for _, r in val_ev.iterrows()], f1=EV_FINAL["arr"])
for _, r in ev_rows[ev_rows.f1 == 0].sample(min(6, (ev_rows.f1 == 0).sum()), random_state=2).iterrows():
    print(f"  risk(actual/pred)={r.risk_level}/{RISK_ORDER[int(pred_risk_va[r.row_id])]} | GOLD={gold_spans(r)} | PRED={[exc(p, 70) for p in r.pred]}")
z = ev_rows[ev_rows.f1 == 0]
cats = {"gold non-empty, predicted empty (missed or gated)": int(sum(1 for _, r in z.iterrows() if gold_spans(r) and not r.pred)),
        "gold non-empty, wrong span predicted": int(sum(1 for _, r in z.iterrows() if gold_spans(r) and r.pred)),
        "gold empty (Indicator), span predicted": int(sum(1 for _, r in z.iterrows() if not gold_spans(r) and r.pred))}
print("\nZero-score evidence rows by failure type:", cats, "| total zero-score rows:", len(z), "of", len(ev_rows))
RESULTS["evidence"]["failure_types"] = cats

## 20. Final Model Selection  (frozen BEFORE any retraining or prediction)

In [ ]:
RESULTS["selected"] = {"risk": RISK_SEL, "evidence": EV_SEL, "factors": FAC_SEL}
with open(os.path.join(OUT_DIR, "final_results.json"), "w") as fh: json.dump(RESULTS, fh, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("Validation results are now frozen in RESULTS / final_results.json. Retraining below cannot change them.\n")
print("Risk     :", RISK_SEL, RESULTS["risk"]["components"]); print("Evidence :", EV_SEL); print("Factors  :", FAC_SEL, RESULTS["factor"]["components"])

## 21. Retraining on ALL labelled data
Selected components are retrained on all 1,635 labelled posts with the hyper-parameters, loss and **epoch count found on validation** (no validation set remains for early stopping).
Blend weights, thresholds and decoding parameters come from the validation stage. The prediction file is read here for the first time as model input.

In [ ]:
t_start = time.time()
CAND_TE = build_cands(test_df); TEST_TEXT = test_df.post_clean
def retrain_transformer(cfg, task, texts, y, num_labels, metric_fn, **kw):
    key = f"FINAL_{task}_{cfg['model']}_lr{cfg['lr']}_{cfg['loss']}_ep{cfg['best_epoch']}_n{len(texts)}"
    return cached(key, lambda: train_transformer(cfg["model"], task, texts, y, test_texts=kw.pop("test_texts", TEST_TEXT), lr=cfg["lr"], epochs=cfg["best_epoch"], fixed_epochs=max(1, cfg["best_epoch"]),
                    micro_bs=cfg["mb"], accum=cfg["acc"], loss_type=cfg["loss"], max_len=cfg["max_len"], seed=SEED, patience=99, metric_fn=metric_fn, num_labels=num_labels, **kw))

# ---- risk ----
y_full_risk = train_df.risk_level.map({r: i for i, r in enumerate(RISK_ORDER)}).values
test_risk_proba = None; test_risk_pred = None; comp = RESULTS["risk"]["components"]
if "SVM" in comp:
    Xf, (Xt,), _ = risk_features(train_df.post_clean_lower, [test_df.post_clean_lower]); test_risk_pred = LinearSVC(C=SVM_C, class_weight="balanced", random_state=SEED, max_iter=5000).fit(Xf, y_full_risk).predict(Xt)
else:
    test_risk_proba = 0
    for name, wgt in comp.items():
        if wgt == 0: continue
        if name == "LR":
            Xf, (Xt,), _ = risk_features(train_df.post_clean_lower, [test_df.post_clean_lower]); m = LogisticRegression(C=1, class_weight="balanced", max_iter=1000, random_state=SEED).fit(Xf, y_full_risk); pr = reorder_proba(m, Xt)
        else:
            print(f">> final retrain RISK {name}"); pr = retrain_transformer(RISK_BEST_BY_ARCH[name], "multiclass", train_df.post_clean, y_full_risk, 4, risk_metric_fn)["test_proba"]
        test_risk_proba = test_risk_proba + wgt * pr
    test_risk_pred = test_risk_proba.argmax(1)
TEST_RISK = [RISK_ORDER[i] for i in test_risk_pred]; print("Predicted risk distribution:", Counter(TEST_RISK))

# ---- factors ----
mlb_full = MultiLabelBinarizer(classes=OFFICIAL_FACTORS); Y_FULL = mlb_full.fit_transform(train_df.factors_unique); comp = RESULTS["factor"]["components"]; test_fac_proba = 0
for name, wgt in comp.items():
    if wgt == 0: continue
    if name == "LR":
        vf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=10000); Xf = vf.fit_transform(train_df.post_clean_lower); pr = ovr_proba(fit_ovr_lr(Xf, Y_FULL), vf.transform(test_df.post_clean_lower))
    else:
        print(f">> final retrain FACTORS {name}"); pr = retrain_transformer(FAC_BEST_BY_ARCH[name], "multilabel", train_df.post_clean, Y_FULL, N_FACTORS, factor_metric_fn, head_lr_mult=FACTOR_HEAD_LR_MULT)["test_proba"]
    test_fac_proba = test_fac_proba + wgt * pr
TEST_FAC_MATRIX = (test_fac_proba >= FAC_THR).astype(int)
TEST_FACTORS = [str(list(l)) for l in mlb_full.inverse_transform(TEST_FAC_MATRIX)]
print("Posts with >=1 predicted factor:", int((TEST_FAC_MATRIX.sum(1) > 0).sum()), "/", len(test_df), "| mean factors/post:", TEST_FAC_MATRIX.sum(1).mean().round(2))

# ---- evidence ----
spec = EV_SYS[EV_SEL]["spec"]; test_pred_risk_map = dict(zip(test_df.row_id, test_risk_pred)); TEST_EVID = []
tr_ev_full = train_df.dropna(subset=["evidence for suicide risk level"])
def ev_lr_full():
    t2, l2 = [], []
    for _, row in tr_ev_full.iterrows():
        g = get_spans(row["evidence for suicide risk level"]); cs = generate_candidates_baseline(row["post_clean"]); t2 += cs; l2 += [1 if any(spans_match(c, x) for x in g if x.lower() != "none") else 0 for c in cs]
    v = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000); X = v.fit_transform([t.lower() for t in t2]); return v, LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED).fit(X, l2)
ev_vec_f, ev_lr_f = ev_lr_full()
if spec is None:        # a first-review extractor won the selection -> reproduce it, but return RAW slices of the post
    for i, (_, r) in enumerate(test_df.iterrows()):
        gated = EV_SEL.startswith("TF-IDF LR + predicted") and TEST_RISK[i] == "Indicator"
        sp = [] if gated else [to_raw(r["post"], x) for x in predict_evidence_baseline(r["post_clean"], ev_vec_f, ev_lr_f, 0.7)]
        TEST_EVID.append([x for x in sp if x])
else:
    ids_te = list(test_df.row_id); non_ind = [i for i in ids_te if RISK_ORDER[int(test_pred_risk_map[i])] != "Indicator"]
    lr_te = {i: ev_lr_f.predict_proba(ev_vec_f.transform([t.lower() for t in CAND_TE[i]["norm"]]))[:, 1] if CAND_TE[i]["norm"] else np.zeros(0) for i in ids_te}
    tf_te = {}
    if spec["kind"] in ("tf", "hybrid"):
        ctx = spec["ctx"]; print(f">> final retrain EVIDENCE scorer (context={ctx})")
        CAND_FULL = build_cands(train_df); gold_risk_full = dict(zip(train_df.row_id, y_full_risk))
        A2, B2, Y2, _ = build_pairs(CAND_FULL, train_df[(train_df.risk_level != "Indicator") & train_df["evidence for suicide risk level"].notna()], ctx, lambda i: int(gold_risk_full[i]))   # ALL labelled non-Indicator posts
        Ate, Bte, _, own = build_pairs(CAND_TE, test_df[test_df.row_id.isin(non_ind)], ctx, lambda i: int(test_pred_risk_map[i]))
        run0 = EV_RUNS[ctx]; cfg = {"model": ev_arch, "lr": EVID_LR, "loss": "wce", "best_epoch": run0["best_epoch"], "mb": ev_mb if ctx else 32, "acc": ev_acc if ctx else 1, "max_len": run0["max_len_used"]}
        key = f"FINAL_evid_{ev_arch}_ctx{int(ctx)}_ep{cfg['best_epoch']}_n{len(A2)}"
        rr = cached(key, lambda: train_transformer(ev_arch, "multiclass", A2, Y2, tr_b=B2, test_texts=Ate, test_b=Bte, lr=EVID_LR, epochs=cfg["best_epoch"], fixed_epochs=max(1, cfg["best_epoch"]), micro_bs=cfg["mb"], accum=cfg["acc"],
                    loss_type="wce", max_len=cfg["max_len"], seed=SEED, patience=99, metric_fn=ev_metric, num_labels=2))
        for i, p in zip(own, rr["test_proba"][:, 1]): tf_te.setdefault(i, []).append(p)
        tf_te = {k: np.array(v) for k, v in tf_te.items()}
    for _, r in test_df.iterrows():
        rid = r["row_id"]
        if rid not in non_ind: TEST_EVID.append([]); continue
        n = len(CAND_TE[rid]["raw"]); z = np.zeros(n)
        sc = lr_te[rid] if spec["kind"] == "lr" else tf_te.get(rid, z) if spec["kind"] == "tf" else spec["w"] * tf_te.get(rid, z) + (1 - spec["w"]) * lr_te[rid]
        TEST_EVID.append(decode(CAND_TE[rid], sc, *spec["params"]))
print(f"Final retraining + prediction time: {(time.time()-t_start)/60:.1f} min")

## 22. Final Prediction Generation

In [ ]:
MAX_EVIDENCE_SPAN_TOKENS = 80
evid_strings = []
for spans in TEST_EVID:
    spans = [s for s in spans if 0 < len(s.split()) <= MAX_EVIDENCE_SPAN_TOKENS]; evid_strings.append("; ".join(s for s in spans))      # spans are raw slices of the post
submission_df = pd.DataFrame({"row_id": test_df["row_id"].values, "risk_level": TEST_RISK, "evidence": evid_strings, "factors": TEST_FACTORS})
SUB_PATH = os.path.join(OUT_DIR, "SSN NeuralForge.csv"); submission_df.to_csv(SUB_PATH, index=False)
print("Saved:", SUB_PATH, submission_df.shape); print("Models used -> risk:", RISK_SEL, "| evidence:", EV_SEL, "| factors:", FAC_SEL); submission_df.head(8)

## 23. Submission Validation

In [ ]:
checks = []
def chk(name, ok, detail=""): checks.append(bool(ok)); print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f" - {detail}" if detail else ""))
back = pd.read_csv(SUB_PATH, keep_default_na=False); post_of = dict(zip(test_df.row_id, test_df.post.astype(str)))
chk("CSV is readable and round-trips", back.shape == submission_df.shape)
chk("column names exactly row_id,risk_level,evidence,factors", list(back.columns) == ["row_id", "risk_level", "evidence", "factors"])
chk("row count equals prediction input", len(back) == len(test_df), f"{len(back)} vs {len(test_df)}")
chk("row_id set equals input row_id set", set(back.row_id) == set(test_df.row_id)); chk("no duplicate row_id", not back.row_id.duplicated().any())
chk("no missing row_id / risk_level / factors", back[["row_id", "risk_level", "factors"]].apply(lambda c: c.astype(str).str.strip().eq("")).sum().sum() == 0)
chk("all risk labels valid", set(back.risk_level) <= set(RISK_ORDER), str(set(back.risk_level) - set(RISK_ORDER) or ""))
bad = 0; tot = 0
for rid, e in zip(back.row_id, back.evidence):
    for s in TEST_EVID[list(test_df.row_id).index(rid)]:
        if not s: continue
        tot += 1; bad += s not in post_of[rid]
chk("every evidence span is an EXACT (case-sensitive) substring of the raw post", bad == 0, f"{bad}/{tot} violations")
ungated = EV_SEL.startswith("First-review")
if ungated: print("[INFO] ungated first-review extractor was selected -> Indicator posts may carry evidence (check skipped)")
else: chk("no evidence for predicted-Indicator posts", (back[back.risk_level == "Indicator"].evidence == "").all())
fx = [ast.literal_eval(f) for f in back.factors]; chk("factors parse as lists of official categories", all(isinstance(l, list) and set(l) <= set(OFFICIAL_FACTORS) for l in fx))
chk("factor lists contain no duplicates", all(len(l) == len(set(l)) for l in fx))
chk("file named exactly 'SSN NeuralForge.csv'", os.path.basename(SUB_PATH) == "SSN NeuralForge.csv")
chk("prediction file never used for tuning (user overlap with train = 0)", len(set(test_df.anon_user_id) & set(train_df.anon_user_id)) == 0)
print("\n" + "=" * 50); print(f"ALL {len(checks)} CHECKS PASSED" if all(checks) else f"{checks.count(False)} OF {len(checks)} CHECKS FAILED"); print("=" * 50)
RESULTS["submission_checks_passed"] = bool(all(checks)); RESULTS["submission_risk_distribution"] = dict(Counter(TEST_RISK))

## 24. Final Summary

In [ ]:
B, Fz = BASELINE, FINAL; d = lambda k: Fz[k] - B[k]; rp = lambda k: d(k) / B[k] * 100
print("=" * 60); print("FINAL REVIEW PERFORMANCE SUMMARY"); print("=" * 60)
print(f"{'':22s}{'FIRST REVIEW':>14s}{'FINAL REVIEW':>16s}"); print("-" * 60)
print(f"{'Risk Weighted F1':22s}{B['risk_weighted_f1']:>14.4f}{Fz['risk_weighted_f1']:>16.4f}")
print(f"{'Evidence Phrase F1':22s}{B['evidence_phrase_f1']:>14.4f}{Fz['evidence_phrase_f1']:>16.4f}")
print(f"{'Factor Macro F1':22s}{B['factor_macro_f1']:>14.4f}{Fz['factor_macro_f1']:>16.4f}"); print("=" * 60)
for k, t in [("risk_weighted_f1", "Risk"), ("evidence_phrase_f1", "Evidence"), ("factor_macro_f1", "Factor")]:
    print(f"\n{t} Improvement:\n{d(k):+.4f} F1\n{rp(k):+.2f}%")
print("\n" + "=" * 60); print("BEST FINAL MODELS"); print("=" * 60)
print(f"Risk:\n{RISK_SEL}  {RESULTS['risk']['components']}\n\nEvidence:\n{EV_SEL}\n\nFactors:\n{FAC_SEL}  {RESULTS['factor']['components']}"); print("=" * 60)
print("\nStatistical caution (paired user-level bootstrap, final - first-review model):")
for k in ("risk", "evidence", "factor"):
    b = RESULTS["bootstrap"][k]; print(f"  {k:9s} mean {b['mean_diff']:+.4f}  95% CI [{b['ci_low']:+.4f}, {b['ci_high']:+.4f}]  P(better)={b['p_better']:.2f}  -> {'distinguishable from noise' if b['ci_low'] > 0 else 'NOT distinguishable from split noise at 95%'}")
for k, tgt in [("risk_weighted_f1", 0.70), ("evidence_phrase_f1", 0.60), ("factor_macro_f1", 0.50)]: print(f"  target {LABELS[k]} > {tgt}: {'reached' if Fz[k] > tgt else 'NOT reached'} ({Fz[k]:.4f})")
with open(os.path.join(OUT_DIR, "final_results.json"), "w") as fh: json.dump(RESULTS, fh, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

In [ ]:
# ============ # FINAL REVIEW — PERFORMANCE IMPROVEMENT  (mentor-facing, generated from the measured numbers) ============
from IPython.display import Markdown, display
tr_won = [k for k in ("risk", "factor") if RESULTS[k]["selected"] not in ("TF-IDF + Logistic Regression", "TF-IDF + Linear SVM", "TF-IDF + OvR Logistic Regression")]
why = []
lr_s, tf_best = RISK_CANDS["TF-IDF + Logistic Regression"]["score"], max([c["score"] for c in RISK_CANDS.values() if c["kind"] == "transformer"] or [float("nan")])
if tf_best == tf_best: why.append(f"**Pretrained Transformers** replaced bag-of-n-grams: best single Transformer risk F1 = {tf_best:.4f} vs TF-IDF LR {lr_s:.4f} on the identical user-grouped split." + (" (blending with TF-IDF added more.)" if RISK_SEL.startswith("Ensemble") else ""))
ftf = max([c["score"] for c in FAC_CANDS.values() if c["kind"] == "transformer"] or [float("nan")])
if ftf == ftf: why.append(f"**Multi-label factors:** pos-weighted sigmoid heads + per-factor thresholds (0.10-0.90) gave Transformer Macro F1 = {ftf:.4f} vs classical {lr_fac_metrics['macro_f1']:.4f}; selected: {FAC_SEL}.")
why.append(f"**Evidence:** the risk-aware gate (no span for predicted *Indicator*) plus candidate re-scoring moved Phrase F1 from {E0:.4f} to {EV_FINAL['score']:.4f}; the gate alone accounts for {E1 - E0:+.4f} of that (see attribution table).")
md_txt = f"""# FINAL REVIEW — PERFORMANCE IMPROVEMENT
During the first review, the system used classical TF-IDF-based models. During the final review, the system was improved using pretrained Transformer models, class-imbalance handling, threshold optimisation, ensemble learning, and improved evidence extraction. All numbers below are measured on the same fixed, user-grouped validation split.

### Risk Classification
{B['risk_weighted_f1']:.4f} → **{Fz['risk_weighted_f1']:.4f}**  ({rp('risk_weighted_f1'):+.2f}%)  — {RISK_SEL}
### Evidence Extraction
{B['evidence_phrase_f1']:.4f} → **{Fz['evidence_phrase_f1']:.4f}**  ({rp('evidence_phrase_f1'):+.2f}%)  — {EV_SEL}
### Factor Identification
{B['factor_macro_f1']:.4f} → **{Fz['factor_macro_f1']:.4f}**  ({rp('factor_macro_f1'):+.2f}%)  — {FAC_SEL}

**Why it changed**
""" + "\n".join(f"- {w}" for w in why) + f"""

**Caveats stated up-front:** validation has ~{val_split.anon_user_id.nunique()} users, so see the bootstrap intervals above; thresholds/decoder are tuned on validation (cross-fitted factor Macro F1 = {RESULTS['factor']['crossfit_macro']:.4f}); evidence Phrase F1 is the local approximation, not the official scorer."""
open(os.path.join(OUT_DIR, "MENTOR_SUMMARY.md"), "w").write(md_txt); display(Markdown(md_txt))

## Reproducibility record

In [ ]:
REPRO = dict(ENV); REPRO.update({"primary_split_seed": PRIMARY_SPLIT_SEED, "risk_grid": [(a, b, c, d) for a, b, c, d in RISK_GRID], "risk_epochs_max": RISK_EPOCHS, "factor_epochs_max": FACTOR_EPOCHS})
RESULTS["reproducibility"] = REPRO; print(json.dumps(REPRO, indent=2, default=str))

## Generated deliverables — README.md and the PDF report (written from the measured results above)

In [ ]:
def _cfg_lines():
    out = []
    for grp, best in (("risk", RISK_BEST_BY_ARCH), ("factors", FAC_BEST_BY_ARCH)):
        for comp in RESULTS["risk" if grp == "risk" else "factor"]["components"]:
            r = best.get(comp)
            if r: out.append(f"{grp}: {r['model']} | lr={r['lr']} | batch={r['batch']} | loss={r['loss']} | max_len={r['max_len']} | best epoch={r['best_epoch']} | seed={r['seed']}")
    return out
CFG_LINES = _cfg_lines(); RESULTS["reproducibility"]["selected_configs"] = CFG_LINES
imp = {k: (FINAL[k] - BASELINE[k], (FINAL[k] - BASELINE[k]) / BASELINE[k] * 100) for k in BASELINE}
tbl_md = "| Metric | First Review | Final Review | Absolute | Relative |\n|---|---|---|---|---|\n" + "\n".join(
    f"| {LABELS[k]} | {BASELINE[k]:.4f} | {FINAL[k]:.4f} | {imp[k][0]:+.4f} | {imp[k][1]:+.2f}% |" for k in BASELINE)
ran_tf = any("failed" not in r for r in RISK_RUNS + FAC_RUNS)
README = f"""# SSN NeuralForge — IEEE BigData 2026 Explainable Suicide Risk Detection (FINAL REVIEW)

## Final architecture
* **Risk (4-class):** selected = `{RISK_SEL}` (components/weights: {RESULTS['risk']['components']}). Candidates compared: TF-IDF+LogReg, TF-IDF+LinearSVM, DistilBERT, RoBERTa, DeBERTa-v3 (those that ran), and blends.
* **Evidence:** selected = `{EV_SEL}`. Risk-aware gate (no span for predicted *Indicator*), candidate sentences/clauses scored, spans returned as exact slices of the original post.
* **Factors (24, multi-label):** selected = `{FAC_SEL}` (components/weights: {RESULTS['factor']['components']}) with per-factor thresholds searched on 0.10-0.90.

## First review (frozen) vs final review (measured on the same user-grouped primary validation split)
{tbl_md}

Statistical caution (paired user-level bootstrap, 1000 resamples, final minus first-review model):
risk {RESULTS['bootstrap']['risk']['mean_diff']:+.4f} [{RESULTS['bootstrap']['risk']['ci_low']:+.4f}, {RESULTS['bootstrap']['risk']['ci_high']:+.4f}],
evidence {RESULTS['bootstrap']['evidence']['mean_diff']:+.4f} [{RESULTS['bootstrap']['evidence']['ci_low']:+.4f}, {RESULTS['bootstrap']['evidence']['ci_high']:+.4f}],
factors {RESULTS['bootstrap']['factor']['mean_diff']:+.4f} [{RESULTS['bootstrap']['factor']['ci_low']:+.4f}, {RESULTS['bootstrap']['factor']['ci_high']:+.4f}].
Cross-fitted factor Macro F1 (thresholds tuned on half of the validation users, scored on the other half): {RESULTS['factor']['crossfit_macro']:.4f}.

## Selected configurations
{chr(10).join('* ' + l for l in CFG_LINES) or '* classical models only'}

## GPU requirements
Transformer sections need a CUDA GPU (Colab T4 16 GB or Kaggle T4/P100 is sufficient; DeBERTa-v3 uses micro-batch 8 x accumulation 2 and fp16).
Allow roughly 1-2 hours for the complete notebook on a T4 (all grids). Finished runs are cached in `outputs/cache/`, so a restart resumes instead of retraining.

## How to run
**Colab:** Runtime -> Change runtime type -> T4 GPU. Upload `train.xlsx` and `final_prediction_input.xlsx` (to `/content` or `/content/data`) and the notebook. Runtime -> Run all.
**Kaggle:** New notebook -> Accelerator GPU T4/P100, Internet ON (needed to download model weights). Add `train.xlsx` and `final_prediction_input.xlsx` as a Dataset. Upload `SSN_NeuralForge.ipynb`. Run All. Outputs appear in `/kaggle/working/SSN_NeuralForge_FINAL/outputs`.
**Local:** place the two Excel files in `../data/`, `pip install -r requirements.txt`, run the notebook (a CUDA GPU is required for the Transformer sections).

## Output files
`outputs/SSN NeuralForge.csv` (submission), `outputs/final_results.json`, `outputs/model_comparison.csv`, `outputs/MENTOR_SUMMARY.md`, figures (`first_vs_final.png`, `risk_confusion_matrix.png`, `factor_f1_comparison.png`, ...), `report/SSN_NeuralForge_Report.pdf`.

## Limitations
* Validation holds only {val_split.anon_user_id.nunique()} users / {len(val_split)} posts: single-split differences of a few points can be noise (see bootstrap intervals above).
* Per-factor thresholds, blend weights and evidence decoding parameters are tuned on the validation split; the cross-fitted estimate quantifies that optimism for factors.
* Evidence Phrase F1 is a local approximation of the official metric (case-insensitive containment either direction, 3x length cap); a strict token-level variant is reported as a sanity check
  (first review {RESULTS['evidence']['strict_first_review']:.4f} -> final {RESULTS['evidence']['strict_final'] if RESULTS['evidence']['strict_final'] is not None else float('nan'):.4f}).
* Candidate spans are sentences/clauses; the oracle ceiling of this candidate set is {RESULTS['evidence']['oracle_ceiling']:.4f}.
* Final models are retrained on all labelled data with the epoch count found on validation; thresholds from the 80 % model are reused.
* Rare factors (< 30 training posts) remain hard.

## Reproducibility
Python {ENV['python']}, PyTorch {ENV['torch']}, Transformers {ENV['transformers']}, scikit-learn {ENV['sklearn']}, GPU {ENV['gpu']} ({ENV['gpu_memory']}), seed {SEED}, primary split seed {PRIMARY_SPLIT_SEED}, max length {MAX_LEN}, fp16 autocast {ENV['amp']}.
Transformer training on GPU with fp16 is not bit-wise deterministic; expect small run-to-run differences.
"""
open(os.path.join(CODE_DIR, "README.md"), "w").write(README); open(os.path.join(OUT_DIR, "README_generated.md"), "w").write(README)
print("README.md written to", os.path.abspath(CODE_DIR))

In [ ]:
# ---------------- PDF report (20 sections, ACTUAL measured numbers) ----------------
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak
from xml.sax.saxutils import escape
ss = getSampleStyleSheet(); body = ParagraphStyle("b", parent=ss["BodyText"], fontSize=9.5, leading=13); h1 = ss["Heading2"]
def P(t): return Paragraph(escape(t).replace("\n", "<br/>"), body)
def T(rows, widths=None):
    t = Table(rows, colWidths=widths, repeatRows=1); t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2a6f97")), ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey), ("FONTSIZE", (0, 0), (-1, -1), 8), ("VALIGN", (0, 0), (-1, -1), "TOP")])); return t
def fig(name, w=15):
    p = os.path.join(OUT_DIR, name)
    return [Image(p, width=w * cm, height=w * cm * (lambda im: im.getbbox()[3] / im.getbbox()[2])(__import__("PIL.Image", fromlist=["Image"]).open(p))), Spacer(1, 6)] if os.path.exists(p) else []
sel_tf = RISK_SEL not in ("TF-IDF + Logistic Regression", "TF-IDF + Linear SVM")
S = []
S += [Paragraph("Explainable Suicide Risk Detection: Final Review Report", ss["Title"]), P("Team SSN NeuralForge - SSN College of Engineering - IEEE BigData 2026 Challenge"), Spacer(1, 8)]
def sec(n, title, *paras, extra=()):
    S.append(Paragraph(f"{n}. {title}", h1)); [S.append(P(p)) for p in paras]; S.extend(extra); S.append(Spacer(1, 6))
b = RESULTS["bootstrap"]
sec(1, "Abstract", f"We improve the first-review TF-IDF pipeline for the IEEE BigData 2026 Explainable Suicide Risk Detection Challenge using pretrained Transformers, class-imbalance handling, per-factor threshold optimisation, probability blending and a risk-aware verbatim evidence extractor. "
    f"On the same fixed user-grouped validation split, risk Weighted F1 moved from {BASELINE['risk_weighted_f1']:.4f} to {FINAL['risk_weighted_f1']:.4f} ({imp['risk_weighted_f1'][1]:+.2f}%), evidence Phrase F1 from {BASELINE['evidence_phrase_f1']:.4f} to {FINAL['evidence_phrase_f1']:.4f} ({imp['evidence_phrase_f1'][1]:+.2f}%), "
    f"and factor Macro F1 from {BASELINE['factor_macro_f1']:.4f} to {FINAL['factor_macro_f1']:.4f} ({imp['factor_macro_f1'][1]:+.2f}%). All values are computed from validation predictions of models executed in the final notebook.")
sec(2, "Introduction", "Suicide-risk assessment from social media text needs both accurate severity classification and interpretable, text-grounded justification. The challenge asks for a four-level risk label with verbatim evidence (Subtask 1) and 24-category multi-label factor identification (Subtask 2).")
sec(3, "Problem Statement", "Subtask 1: classify a post as Indicator, Ideation, Behavior or Attempt (Weighted F1) and extract short verbatim evidence (Phrase F1). Subtask 2: predict all applicable factors from 24 categories (Macro F1).")
sec(4, "Dataset", f"train.xlsx: {len(train_df)} labelled posts from {train_df.anon_user_id.nunique()} users. Prediction file: {len(test_df)} unlabelled posts from {test_df.anon_user_id.nunique()} users, with zero user overlap with training. Primary validation: {len(val_split)} posts / {val_split.anon_user_id.nunique()} users.", extra=fig("dataset_statistics.png", 16))
sec(5, "Preprocessing", "Risk labels were case-normalised; whitespace collapsed; punctuation and negations kept. Factor strings parsed with ast.literal_eval and deduplicated. Evidence candidates keep character offsets into the raw post.")
sec(6, "Baseline Approach", "Risk: word+char TF-IDF + Logistic Regression. Evidence: sentence/clause candidates scored by TF-IDF LR (threshold 0.7). Factors: word TF-IDF + OvR LR with per-factor thresholds.")
sec(7, "First Review Results", f"Risk Weighted F1 {BASELINE['risk_weighted_f1']:.4f}; Evidence Phrase F1 {BASELINE['evidence_phrase_f1']:.4f}; Factor Macro F1 {BASELINE['factor_macro_f1']:.4f}. These were reproduced exactly by the final notebook on the same split ({RESULTS['repro_risk']:.4f} / {RESULTS['repro_evidence']:.4f} / {RESULTS['repro_factor']:.4f}).")
sec(8, "Proposed Final Approach", f"Selected risk model: {RISK_SEL}. Selected evidence system: {EV_SEL}. Selected factor model: {FAC_SEL}. Selection policy: highest validated score on the primary split; models within {SELECTION_TOLERANCE} of the best are tie-broken toward the simpler one; no test-set information is used.", extra=fig("pipeline_diagram.png", 17))
sec(9, "Transformer Architecture", "Candidates: DistilBERT-base, RoBERTa-base, DeBERTa-v3-base with a sequence-classification head (softmax for risk, sigmoid with problem_type=multi_label_classification for factors, 2-way head for evidence candidates).\n" + "\n".join(f"Ran: {r['model']} lr={r.get('lr')} -> {r.get('best_score', float('nan')):.4f}" for r in RISK_RUNS if 'failed' not in r) + ("\nFailed: " + "; ".join(f"{r['model']} ({r['failed'][:60]})" for r in RISK_RUNS + FAC_RUNS if 'failed' in r) if any('failed' in r for r in RISK_RUNS + FAC_RUNS) else ""))
sec(10, "Training Strategy", f"AdamW, weight decay 0.01, linear warm-up (10%) and decay, gradient clipping 1.0, fp16 autocast, early stopping with best-checkpoint restore, max length {MAX_LEN}, fixed seed {SEED}. Class imbalance: class-weighted CE / focal for risk; sqrt-scaled pos-weight BCE / focal for factors; the loss that validated best was kept.")
rk = [["Model", "LR", "Loss", "Val Weighted F1", "Best epoch", "Time (s)"]] + [[r["model"], str(r.get("lr")), r.get("loss", ""), f"{r.get('best_score', float('nan')):.4f}", str(r.get("best_epoch")), f"{r.get('seconds', 0):.0f}"] for r in RISK_RUNS if "failed" not in r]
sec(11, "Hyperparameter Optimization", "Grid over architecture and learning rate on the fixed split, followed by a loss ablation on the best configuration.", extra=[T(rk)])
sec(12, "Evidence Extraction", f"Candidates are sentences and clauses taken as character slices of the raw post, so every output is a verbatim substring. Predicted Indicator posts receive no evidence (gold evidence is 'none' for ~95% of Indicator posts). Selected: {EV_SEL}. Candidate oracle ceiling {RESULTS['evidence']['oracle_ceiling']:.4f}. Strict token-level sanity metric: {RESULTS['evidence']['strict_first_review']:.4f} -> {(RESULTS['evidence']['strict_final'] or float('nan')):.4f}.",
    extra=[T([["System", "Phrase F1"]] + [[k, f"{v:.4f}"] for k, v in RESULTS["evidence"]["systems"].items()])])
sec(13, "Multi-label Factor Classification", f"Sigmoid heads, pos-weighted BCE (focal tested), early stopping on mean average precision (threshold-free), then per-factor thresholds on 0.10-0.90. Final Macro F1 {FAC_FINAL['macro_f1']:.4f}, Micro F1 {FAC_FINAL['micro_f1']:.4f}, Hamming loss {FAC_FINAL['hamming_loss']:.4f}, exact match {FAC_FINAL['exact_match']:.4f}.", extra=fig("factor_f1_comparison.png", 14))
sec(14, "Ensemble Strategy", "Probabilities of the TF-IDF model and the Transformer(s) are blended with weights searched on validation (0.0-1.0, step 0.1); factor blends are scored after their own threshold optimisation. Ensembles were adopted only when the selection policy favoured them.")
cand_rows = [["Task", "Model", "Score"]] + [["Risk", n, f"{c['score']:.4f}"] for n, c in RISK_CANDS.items()] + [["Factors", n, f"{c['score']:.4f}"] for n, c in FAC_CANDS.items()]
sec(15, "Experimental Results", "All candidates, primary validation split:", extra=[T(cand_rows)])
sec(16, "First vs Final Comparison", "Paired user-level bootstrap (1000 resamples) of final minus first-review model: " + "; ".join(f"{k} {b[k]['mean_diff']:+.4f} (95% CI {b[k]['ci_low']:+.4f}..{b[k]['ci_high']:+.4f})" for k in ("risk", "evidence", "factor")) + ".",
    extra=[T([["Metric", "First", "Final", "Abs.", "Rel. %"]] + [[LABELS[k], f"{BASELINE[k]:.4f}", f"{FINAL[k]:.4f}", f"{imp[k][0]:+.4f}", f"{imp[k][1]:+.2f}"] for k in BASELINE])] + fig("first_vs_final.png", 17))
sec(17, "Error Analysis", f"Risk errors concentrate on adjacent severity levels; evidence zero-score rows by type: {RESULTS['evidence'].get('failure_types')}. Rare factors with few training posts remain weak.", extra=fig("risk_confusion_matrix.png", 9))
sec(18, "Discussion", "The attribution table shows which part of the evidence gain comes from the risk gate versus better scoring. Thresholds, blend weights and decoder settings are tuned on the validation split; cross-fitted factor Macro F1 is " + f"{RESULTS['factor']['crossfit_macro']:.4f} (classical model: {RESULTS['factor']['crossfit_macro_baseline_model']:.4f}).")
sec(19, "Limitations", f"Small validation set ({val_split.anon_user_id.nunique()} users); validation-tuned thresholds and decoders; local approximation of the evidence metric; leaderboard performance may differ; rare factors under-represented.")
sec(20, "Conclusion", f"Final measured scores: risk {FINAL['risk_weighted_f1']:.4f}, evidence {FINAL['evidence_phrase_f1']:.4f}, factors {FINAL['factor_macro_f1']:.4f} versus first review {BASELINE['risk_weighted_f1']:.4f} / {BASELINE['evidence_phrase_f1']:.4f} / {BASELINE['factor_macro_f1']:.4f}. Submission file validated ({'all checks passed' if RESULTS.get('submission_checks_passed') else 'CHECK FAILURES - see notebook'}).")
S.append(Paragraph("References", h1)); S.append(P("[1] IEEE BigData 2026 Explainable Suicide Risk Detection Challenge. [2] Pedregosa et al., scikit-learn, JMLR 2011. [3] Sanh et al., DistilBERT, 2019. [4] Liu et al., RoBERTa, 2019. [5] He et al., DeBERTaV3, 2021. [6] Wolf et al., Transformers, EMNLP-Demos 2020."))
rep_path = os.path.join(REPORT_DIR, "SSN_NeuralForge_Report.pdf"); SimpleDocTemplate(rep_path, pagesize=A4, leftMargin=2 * cm, rightMargin=2 * cm, topMargin=2 * cm, bottomMargin=2 * cm).build(S)
print("Report written:", os.path.abspath(rep_path))
with open(os.path.join(OUT_DIR, "final_results.json"), "w") as fh: json.dump(RESULTS, fh, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))